# Explanation-Guided Radiomics with Multi-Method XAI Validation
# for Colon Cancer Classification in Abdominal CT

## Complete Jupyter Notebook Study Implementation

**Dataset:** [Kaggle ColonCancerCT-2025](https://www.kaggle.com/datasets/orvile/coloncancerct-2025-abdominal-ct-scans)

---

### Study Contributions:
1. **Multi-XAI Comparison** on abdominal CT classification
2. **Quantitative XAI Validation** with faithfulness and sanity checks
3. **Explanation-Guided Focal Radiomics** extraction and fusion

### Instructions:
1. Download dataset from Kaggle
2. Update  path in the Configuration cell
3. Run all cells sequentially


In [ ]:
# Run this cell first to install all dependencies
!pip install torch torchvision --quiet
!pip install opencv-python pillow scikit-learn pandas numpy matplotlib seaborn --quiet
!pip install grad-cam captum shap lime --quiet
!pip install scikit-image --quiet
!pip install timm --quiet
!pip install xgboost --quiet
!pip install tqdm albumentations --quiet
!pip install scipy --quiet

print("All packages installed successfully!")

In [ ]:
"""
============================================================================
EXPLANATION-GUIDED RADIOMICS WITH MULTI-METHOD XAI VALIDATION
FOR COLON CANCER CLASSIFICATION IN ABDOMINAL CT
============================================================================
Part 1: Setup, Data Loading, and Preprocessing

Author: [Your Name]
Date: 2025
Dataset: https://www.kaggle.com/datasets/orvile/coloncancerct-2025-abdominal-ct-scans
============================================================================
"""


In [ ]:
# CELL 1: INSTALL REQUIRED PACKAGES


In [ ]:
# Run this cell first to install all dependencies

"""
!pip install torch torchvision --quiet
!pip install opencv-python pillow scikit-learn pandas numpy matplotlib seaborn --quiet
!pip install grad-cam captum shap lime --quiet
!pip install scikit-image pyradiomics SimpleITK --quiet
!pip install efficientnet_pytorch timm --quiet
!pip install xgboost lightgbm --quiet
!pip install tqdm albumentations --quiet
!pip install scipy statsmodels --quiet

print("All packages installed successfully!")
"""


In [ ]:
!pip install grad-cam captum shap lime --quiet


In [ ]:
!pip install scikit-image --quiet
!pip install timm --quiet
!pip install xgboost --quiet
!pip install tqdm albumentations --quiet
!pip install scipy --quiet

In [ ]:
!pip install grad-cam captum shap lime --quiet
!pip install scikit-image pyradiomics SimpleITK --quiet
!pip install efficientnet_pytorch timm --quiet
!pip install xgboost lightgbm --quiet
!pip install tqdm albumentations --quiet
!pip install scipy statsmodels --quiet

In [ ]:
# CELL 2: IMPORT LIBRARIES


In [ ]:
import os
import sys
import random
import warnings
import json
from pathlib import Path
from collections import defaultdict
from datetime import datetime

# Data handling
import numpy as np
import pandas as pd
from PIL import Image
import cv2

# Deep Learning
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision
from torchvision import transforms, models
import timm

# XAI Libraries
from pytorch_grad_cam import GradCAM, GradCAMPlusPlus, ScoreCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image
from captum.attr import IntegratedGradients, Saliency, DeepLift, GradientShap
from captum.attr import visualization as viz
import shap
from lime import lime_image
from skimage.segmentation import mark_boundaries

# Radiomics
from radiomics import featureextractor
import SimpleITK as sitk

# Machine Learning
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, roc_curve, confusion_matrix, classification_report,
    precision_recall_curve, average_precision_score
)
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import xgboost as xgb

# Visualization
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from matplotlib.colors import LinearSegmentedColormap

# Statistics
from scipy import stats
from scipy.ndimage import label as scipy_label
from scipy.ndimage import binary_dilation, binary_erosion

# Augmentation
import albumentations as A
from albumentations.pytorch import ToTensorV2

# Progress bar
from tqdm.auto import tqdm

# Suppress warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


In [ ]:
# CELL 3: CONFIGURATION


In [ ]:
class Config:
    """Configuration class for the entire study"""
    
    # ========================================
    # MODIFY THESE PATHS TO MATCH YOUR SETUP
    # ========================================
    DATA_ROOT = './Split_Data'  # Root folder containing train/val/test
    TRAIN_DIR = os.path.join(DATA_ROOT, "train")
    VAL_DIR = os.path.join(DATA_ROOT, "val")
    TEST_DIR = os.path.join(DATA_ROOT, "test")
    
    # Class names (modify based on your folder names)
    CLASSES = ['Non_Cancer', 'cancer']
    NUM_CLASSES = 2
    
    # Image settings
    IMG_SIZE = 224
    IMG_CHANNELS = 3
    
    # Training settings
    BATCH_SIZE = 32
    NUM_EPOCHS = 30
    LEARNING_RATE = 1e-4
    WEIGHT_DECAY = 1e-4
    PATIENCE = 7  # Early stopping patience
    
    # Model settings
    MODEL_NAME = 'efficientnet_b0'  # Options: 'resnet50', 'efficientnet_b0', 'resnet18'
    PRETRAINED = True
    
    # XAI settings
    XAI_METHODS = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
    HEATMAP_THRESHOLD = 0.15  # Top 15% activations for focal ROI
    
    # Radiomics settings
    RADIOMICS_BIN_WIDTH = 25
    
    # Output settings
    OUTPUT_DIR = "./results"
    MODEL_SAVE_PATH = os.path.join(OUTPUT_DIR, "models")
    FIGURE_SAVE_PATH = os.path.join(OUTPUT_DIR, "figures")
    
    @classmethod
    def setup_dirs(cls):
        os.makedirs(cls.OUTPUT_DIR, exist_ok=True)
        os.makedirs(cls.MODEL_SAVE_PATH, exist_ok=True)
        os.makedirs(cls.FIGURE_SAVE_PATH, exist_ok=True)

# Setup directories
Config.setup_dirs()
print("Configuration loaded!")
print(f"Data root: {Config.DATA_ROOT}")
print(f"Image size: {Config.IMG_SIZE}x{Config.IMG_SIZE}")
print(f"Model: {Config.MODEL_NAME}")


In [ ]:
# CELL 4: DATASET EXPLORATION


In [ ]:
def explore_dataset(data_dir):
    """Explore and summarize the dataset structure"""
    
    dataset_info = {}
    
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(data_dir, split)
        if not os.path.exists(split_dir):
            print(f"Warning: {split_dir} does not exist!")
            continue
            
        split_info = {}
        for class_name in os.listdir(split_dir):
            class_dir = os.path.join(split_dir, class_name)
            if os.path.isdir(class_dir):
                images = [f for f in os.listdir(class_dir) 
                         if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff'))]
                split_info[class_name] = len(images)
        
        dataset_info[split] = split_info
    
    return dataset_info


def print_dataset_summary(dataset_info):
    """Print formatted dataset summary"""
    print("=" * 50)
    print("DATASET SUMMARY")
    print("=" * 50)
    
    total_images = 0
    for split, classes in dataset_info.items():
        print(f"\n{split.upper()} SET:")
        split_total = 0
        for class_name, count in classes.items():
            print(f"  {class_name}: {count} images")
            split_total += count
        print(f"  Total: {split_total} images")
        total_images += split_total
    
    print(f"\n{'=' * 50}")
    print(f"TOTAL IMAGES: {total_images}")
    print("=" * 50)


# Explore dataset
print("Exploring dataset structure...\n")
dataset_info = explore_dataset(Config.DATA_ROOT)
print_dataset_summary(dataset_info)


In [ ]:
# CELL 5: VISUALIZE SAMPLE IMAGES


In [ ]:
def visualize_samples(data_dir, num_samples=3, save_path=None):
    """Visualize sample images from each class"""
    
    fig, axes = plt.subplots(2, num_samples, figsize=(15, 10))
    
    train_dir = os.path.join(data_dir, 'train')
    
    if not os.path.exists(train_dir):
        print(f"Train directory not found: {train_dir}")
        return
    
    classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
    
    for i, class_name in enumerate(classes[:2]):
        class_dir = os.path.join(train_dir, class_name)
        
        images = [f for f in os.listdir(class_dir) 
                 if f.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp'))]
        
        sample_images = random.sample(images, min(num_samples, len(images)))
        
        for j, img_name in enumerate(sample_images):
            img_path = os.path.join(class_dir, img_name)
            img = Image.open(img_path).convert('RGB')
            
            axes[i, j].imshow(img, cmap='gray' if img.mode == 'L' else None)
            axes[i, j].set_title(f"{class_name}\n{img_name[:20]}...", fontsize=10)
            axes[i, j].axis('off')
    
    plt.suptitle("Sample CT Images from Dataset", fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Visualize samples
visualize_samples(Config.DATA_ROOT, save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'sample_images.png'))


In [ ]:
# CELL 6: CUSTOM DATASET CLASS


In [ ]:
class CTImageDataset(Dataset):
    """Custom Dataset for CT Images"""
    
    def __init__(self, data_dir, transform=None):
        self.data_dir = data_dir
        self.transform = transform
        
        self.images = []
        self.labels = []
        self.class_to_idx = {}
        self.idx_to_class = {}
        
        self._load_data()
        
    def _load_data(self):
        """Load image paths and labels"""
        classes = sorted([d for d in os.listdir(self.data_dir) 
                         if os.path.isdir(os.path.join(self.data_dir, d))])
        
        for idx, class_name in enumerate(classes):
            class_dir = os.path.join(self.data_dir, class_name)
            
            self.class_to_idx[class_name] = idx
            self.idx_to_class[idx] = class_name
            
            for img_name in os.listdir(class_dir):
                if img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.tif', '.tiff')):
                    self.images.append(os.path.join(class_dir, img_name))
                    self.labels.append(idx)
        
        print(f"Loaded {len(self.images)} images from {len(self.class_to_idx)} classes")
        print(f"Class mapping: {self.class_to_idx}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.labels[idx]
        
        # Load image
        image = Image.open(img_path).convert('RGB')
        image = np.array(image)
        
        if self.transform:
            transformed = self.transform(image=image)
            image = transformed['image']
        
        return image, label, img_path


In [ ]:
# CELL 7: DATA AUGMENTATION


In [ ]:
def get_transforms(img_size, augment=False):
    """Get data transforms using Albumentations"""
    
    if augment:
        # Strong augmentation for training (important for small datasets)
        transform = A.Compose([
            A.Resize(img_size, img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=15, p=0.5),
            A.OneOf([
                A.CLAHE(clip_limit=2.0, p=1),
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=1),
                A.RandomGamma(gamma_limit=(80, 120), p=1),
            ], p=0.5),
            A.OneOf([
                A.GaussNoise(var_limit=(10, 50), p=1),
                A.GaussianBlur(blur_limit=(3, 7), p=1),
            ], p=0.3),
            A.CoarseDropout(max_holes=8, max_height=16, max_width=16, p=0.3),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    else:
        # Validation/Test transform
        transform = A.Compose([
            A.Resize(img_size, img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2(),
        ])
    
    return transform


In [ ]:
# CELL 8: CREATE DATA LOADERS


In [ ]:
def create_data_loaders(config):
    """Create train, validation, and test data loaders"""
    
    # Get transforms
    train_transform = get_transforms(config.IMG_SIZE, augment=True)
    val_transform = get_transforms(config.IMG_SIZE, augment=False)
    
    # Create datasets
    train_dataset = CTImageDataset(config.TRAIN_DIR, transform=train_transform)
    val_dataset = CTImageDataset(config.VAL_DIR, transform=val_transform)
    test_dataset = CTImageDataset(config.TEST_DIR, transform=val_transform)
    
    # Calculate class weights for imbalanced data
    train_labels = train_dataset.labels
    class_counts = np.bincount(train_labels)
    class_weights = 1.0 / class_counts
    sample_weights = class_weights[train_labels]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset, 
        batch_size=config.BATCH_SIZE,
        sampler=sampler,
        num_workers=0,
        pin_memory=True
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    test_loader = DataLoader(
        test_dataset,
        batch_size=config.BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True
    )
    
    return train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset


# Create data loaders
print("Creating data loaders...")
train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset = create_data_loaders(Config)

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset)} samples")
print(f"  Validation: {len(val_dataset)} samples")
print(f"  Test: {len(test_dataset)} samples")


In [ ]:
# CELL 9: VISUALIZE AUGMENTATIONS


In [ ]:
def visualize_augmentations(dataset, num_samples=4, save_path=None):
    """Visualize augmentation effects on sample images"""
    
    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4*num_samples))
    
    aug_transform = get_transforms(Config.IMG_SIZE, augment=True)
    
    indices = random.sample(range(len(dataset.images)), num_samples)
    
    for i, idx in enumerate(indices):
        img_path = dataset.images[idx]
        label = dataset.labels[idx]
        class_name = dataset.idx_to_class[label]
        
        orig_img = np.array(Image.open(img_path).convert('RGB'))
        
        # Original
        axes[i, 0].imshow(orig_img)
        axes[i, 0].set_title(f"Original ({class_name})")
        axes[i, 0].axis('off')
        
        # Show 3 different augmentations
        for j in range(1, 4):
            augmented = aug_transform(image=orig_img)['image']
            aug_vis = augmented.permute(1, 2, 0).numpy()
            aug_vis = aug_vis * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
            aug_vis = np.clip(aug_vis, 0, 1)
            
            axes[i, j].imshow(aug_vis)
            axes[i, j].set_title(f"Augmentation {j}")
            axes[i, j].axis('off')
    
    plt.suptitle("Data Augmentation Examples", fontsize=14, fontweight='bold')
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


# Visualize augmentations
visualize_augmentations(train_dataset, save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'augmentation_examples.png'))

print("\n" + "="*50)
print("PART 1 COMPLETE: Data Loading and Preprocessing")
print("="*50)


In [ ]:
"""
============================================================================
PART 2: MODEL ARCHITECTURE AND TRAINING
============================================================================
"""


In [ ]:
# CELL 10: MODEL ARCHITECTURE


In [ ]:
class ColonCancerClassifier(nn.Module):
    """Colon Cancer Classification Model"""
    
    def __init__(self, model_name='efficientnet_b0', num_classes=2, pretrained=True):
        super(ColonCancerClassifier, self).__init__()
        
        self.model_name = model_name
        
        if model_name == 'efficientnet_b0':
            self.backbone = timm.create_model('efficientnet_b0', pretrained=pretrained)
            num_features = self.backbone.classifier.in_features
            self.backbone.classifier = nn.Identity()
            
        elif model_name == 'resnet50':
            self.backbone = models.resnet50(pretrained=pretrained)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
            
        elif model_name == 'resnet18':
            self.backbone = models.resnet18(pretrained=pretrained)
            num_features = self.backbone.fc.in_features
            self.backbone.fc = nn.Identity()
            
        else:
            raise ValueError(f"Unknown model: {model_name}")
        
        # Classification head
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(num_features, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )
        
        self.num_features = num_features
        
    def forward(self, x):
        features = self.backbone(x)
        logits = self.classifier(features)
        return logits
    
    def get_features(self, x):
        """Extract deep features before classifier"""
        return self.backbone(x)
    
    def get_target_layer(self):
        """Get target layer for Grad-CAM"""
        if 'efficientnet' in self.model_name:
            return self.backbone.conv_head
        elif 'resnet' in self.model_name:
            return self.backbone.layer4[-1]
        else:
            raise ValueError(f"Unknown model for target layer: {self.model_name}")


def create_model(config):
    """Create and initialize model"""
    model = ColonCancerClassifier(
        model_name=config.MODEL_NAME,
        num_classes=config.NUM_CLASSES,
        pretrained=config.PRETRAINED
    )
    model = model.to(device)
    return model


# Create model
model = create_model(Config)
print(f"Model created: {Config.MODEL_NAME}")
print(f"Number of parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")


In [ ]:
# CELL 11: TRAINING UTILITIES


In [ ]:
class EarlyStopping:
    """Early stopping to prevent overfitting"""
    
    def __init__(self, patience=10, min_delta=0, restore_best_weights=True):
        self.patience = patience
        self.min_delta = min_delta
        self.restore_best_weights = restore_best_weights
        self.counter = 0
        self.best_loss = None
        self.best_weights = None
        self.early_stop = False
        
    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
                if self.restore_best_weights:
                    model.load_state_dict(self.best_weights)
        else:
            self.best_loss = val_loss
            self.best_weights = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0


def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc='Training')
    
    for images, labels, _ in pbar:
        images = images.to(device)
        labels = labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    return running_loss / len(train_loader), 100. * correct / total


def validate_epoch(model, val_loader, criterion, device):
    """Validate for one epoch"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc='Validating'):
            images = images.to(device)
            labels = labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    
    accuracy = 100. * correct / total
    avg_loss = running_loss / len(val_loader)
    
    return avg_loss, accuracy, all_preds, all_labels, all_probs


In [ ]:
# CELL 12: MAIN TRAINING LOOP


In [ ]:
def train_model(model, train_loader, val_loader, config, device):
    """Main training function"""
    
    # Loss function with class weights
    train_labels = train_dataset.labels
    class_counts = np.bincount(train_labels)
    class_weights = torch.tensor(len(train_labels) / (len(class_counts) * class_counts), dtype=torch.float).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    
    # Optimizer
    optimizer = optim.AdamW(model.parameters(), lr=config.LEARNING_RATE, weight_decay=config.WEIGHT_DECAY)
    
    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.NUM_EPOCHS, eta_min=1e-6)
    
    # Early stopping
    early_stopping = EarlyStopping(patience=config.PATIENCE, restore_best_weights=True)
    
    # Training history
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss': [], 'val_acc': [],
        'lr': []
    }
    
    print(f"\n{'='*60}")
    print(f"Starting Training: {config.MODEL_NAME}")
    print(f"{'='*60}\n")
    
    best_val_acc = 0
    
    for epoch in range(config.NUM_EPOCHS):
        print(f"\nEpoch {epoch+1}/{config.NUM_EPOCHS}")
        print("-" * 40)
        
        # Train
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        
        # Validate
        val_loss, val_acc, _, _, _ = validate_epoch(model, val_loader, criterion, device)
        
        # Update scheduler
        scheduler.step()
        current_lr = optimizer.param_groups[0]['lr']
        
        # Save history
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['lr'].append(current_lr)
        
        print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")
        print(f"Learning Rate: {current_lr:.6f}")
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
            }, os.path.join(config.MODEL_SAVE_PATH, f'best_model_{config.MODEL_NAME}.pth'))
            print(f"★ Best model saved (Val Acc: {val_acc:.2f}%)")
        
        # Early stopping
        early_stopping(val_loss, model)
        if early_stopping.early_stop:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break
    
    print(f"\n{'='*60}")
    print(f"Training Complete! Best Val Acc: {best_val_acc:.2f}%")
    print(f"{'='*60}")
    
    return history


# Train the model
history = train_model(model, train_loader, val_loader, Config, device)


In [ ]:
# CELL 13: PLOT TRAINING HISTORY


In [ ]:
def plot_training_history(history, save_path=None):
    """Plot training and validation metrics"""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Loss
    axes[0].plot(history['train_loss'], label='Train Loss', color='blue', linewidth=2)
    axes[0].plot(history['val_loss'], label='Val Loss', color='red', linewidth=2)
    axes[0].set_xlabel('Epoch', fontsize=12)
    axes[0].set_ylabel('Loss', fontsize=12)
    axes[0].set_title('Training and Validation Loss', fontsize=14, fontweight='bold')
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy
    axes[1].plot(history['train_acc'], label='Train Acc', color='blue', linewidth=2)
    axes[1].plot(history['val_acc'], label='Val Acc', color='red', linewidth=2)
    axes[1].set_xlabel('Epoch', fontsize=12)
    axes[1].set_ylabel('Accuracy (%)', fontsize=12)
    axes[1].set_title('Training and Validation Accuracy', fontsize=14, fontweight='bold')
    axes[1].legend(fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    # Learning Rate
    axes[2].plot(history['lr'], color='green', linewidth=2)
    axes[2].set_xlabel('Epoch', fontsize=12)
    axes[2].set_ylabel('Learning Rate', fontsize=12)
    axes[2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
    axes[2].grid(True, alpha=0.3)
    axes[2].set_yscale('log')
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_training_history(history, save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'training_history.png'))


In [ ]:
# CELL 14: MODEL EVALUATION


In [ ]:
def load_best_model(model, config):
    """Load the best saved model"""
    checkpoint_path = os.path.join(config.MODEL_SAVE_PATH, f'best_model_{config.MODEL_NAME}.pth')
    if os.path.exists(checkpoint_path):
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint['model_state_dict'])
        print(f"Loaded best model from epoch {checkpoint['epoch']+1} with Val Acc: {checkpoint['val_acc']:.2f}%")
    else:
        print("No checkpoint found, using current model")
    return model


def comprehensive_evaluation(model, test_loader, device, class_names):
    """Comprehensive model evaluation"""
    
    model.eval()
    all_preds = []
    all_labels = []
    all_probs = []
    
    with torch.no_grad():
        for images, labels, _ in tqdm(test_loader, desc='Evaluating'):
            images = images.to(device)
            
            outputs = model(images)
            probs = F.softmax(outputs, dim=1)
            _, predicted = outputs.max(1)
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
            all_probs.extend(probs[:, 1].cpu().numpy())
    
    # Calculate metrics
    accuracy = accuracy_score(all_labels, all_preds)
    precision = precision_score(all_labels, all_preds, average='binary', zero_division=0)
    recall = recall_score(all_labels, all_preds, average='binary', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    specificity = recall_score(all_labels, all_preds, average='binary', pos_label=0, zero_division=0)
    
    try:
        auc = roc_auc_score(all_labels, all_probs)
    except ValueError:
        auc = 0.5
    
    # Print results
    print("\n" + "="*60)
    print("TEST SET EVALUATION RESULTS")
    print("="*60)
    print(f"Accuracy:    {accuracy*100:.2f}%")
    print(f"Precision:   {precision*100:.2f}%")
    print(f"Sensitivity: {recall*100:.2f}% (Recall)")
    print(f"Specificity: {specificity*100:.2f}%")
    print(f"F1-Score:    {f1*100:.2f}%")
    print(f"AUC-ROC:     {auc:.4f}")
    print("="*60)
    
    print("\nClassification Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, zero_division=0))
    
    results = {
        'accuracy': accuracy,
        'precision': precision,
        'recall': recall,
        'specificity': specificity,
        'f1': f1,
        'auc': auc,
        'predictions': all_preds,
        'labels': all_labels,
        'probabilities': all_probs
    }
    
    return results


# Load best model and evaluate
model = load_best_model(model, Config)
class_names = list(test_dataset.class_to_idx.keys())
test_results = comprehensive_evaluation(model, test_loader, device, class_names)


In [ ]:
# CELL 15: PLOT EVALUATION METRICS


In [ ]:
def plot_evaluation_metrics(results, class_names, save_path=None):
    """Plot confusion matrix and ROC curve"""
    
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Confusion Matrix
    cm = confusion_matrix(results['labels'], results['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
                xticklabels=class_names, yticklabels=class_names)
    axes[0].set_xlabel('Predicted', fontsize=12)
    axes[0].set_ylabel('Actual', fontsize=12)
    axes[0].set_title('Confusion Matrix', fontsize=14, fontweight='bold')
    
    # ROC Curve
    fpr, tpr, _ = roc_curve(results['labels'], results['probabilities'])
    axes[1].plot(fpr, tpr, color='blue', linewidth=2, label=f'AUC = {results["auc"]:.4f}')
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[1].set_xlabel('False Positive Rate', fontsize=12)
    axes[1].set_ylabel('True Positive Rate', fontsize=12)
    axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
    axes[1].legend(loc='lower right', fontsize=10)
    axes[1].grid(True, alpha=0.3)
    
    # Precision-Recall Curve
    precision_vals, recall_vals, _ = precision_recall_curve(results['labels'], results['probabilities'])
    ap = average_precision_score(results['labels'], results['probabilities'])
    axes[2].plot(recall_vals, precision_vals, color='green', linewidth=2, label=f'AP = {ap:.4f}')
    axes[2].set_xlabel('Recall', fontsize=12)
    axes[2].set_ylabel('Precision', fontsize=12)
    axes[2].set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
    axes[2].legend(loc='lower left', fontsize=10)
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_evaluation_metrics(test_results, class_names, 
                       save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'evaluation_metrics.png'))

print("\n" + "="*50)
print("PART 2 COMPLETE: Model Training and Evaluation")
print("="*50)


In [ ]:
"""
============================================================================
PART 3: XAI METHODS IMPLEMENTATION
============================================================================
"""


In [ ]:
# CELL 16: XAI UTILITY FUNCTIONS


In [ ]:
def denormalize_image(tensor, mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]):
    """Denormalize image tensor for visualization"""
    mean = torch.tensor(mean).view(3, 1, 1)
    std = torch.tensor(std).view(3, 1, 1)
    if tensor.device.type == 'cuda':
        mean = mean.to(tensor.device)
        std = std.to(tensor.device)
    tensor = tensor * std + mean
    return torch.clamp(tensor, 0, 1)


def tensor_to_numpy(tensor):
    """Convert tensor to numpy array for visualization"""
    if tensor.dim() == 4:
        tensor = tensor.squeeze(0)
    tensor = denormalize_image(tensor)
    return tensor.permute(1, 2, 0).cpu().numpy()


def normalize_heatmap(heatmap):
    """Normalize heatmap to [0, 1] range"""
    heatmap = np.array(heatmap)
    heatmap = heatmap - heatmap.min()
    if heatmap.max() > 0:
        heatmap = heatmap / heatmap.max()
    return heatmap


def resize_heatmap(heatmap, target_size):
    """Resize heatmap to target size"""
    return cv2.resize(heatmap, (target_size, target_size), interpolation=cv2.INTER_LINEAR)


In [ ]:
# CELL 17: GRAD-CAM++ EXPLAINER


In [ ]:
class GradCAMExplainer:
    """Grad-CAM and Grad-CAM++ explainer"""
    
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.gradcam = GradCAMPlusPlus(model=model, target_layers=[target_layer])
        
    def explain(self, input_tensor, target_class=None):
        """
        Generate Grad-CAM++ heatmap
        
        Args:
            input_tensor: Input image tensor (1, C, H, W)
            target_class: Target class index (None for predicted class)
        
        Returns:
            heatmap: Normalized heatmap (H, W)
        """
        if target_class is not None:
            targets = [ClassifierOutputTarget(target_class)]
        else:
            targets = None
        
        grayscale_cam = self.gradcam(input_tensor=input_tensor, targets=targets)
        heatmap = grayscale_cam[0, :]
        
        return normalize_heatmap(heatmap)
    
    def visualize(self, input_tensor, heatmap):
        """Overlay heatmap on original image"""
        rgb_img = tensor_to_numpy(input_tensor)
        visualization = show_cam_on_image(rgb_img, heatmap, use_rgb=True)
        return visualization


In [ ]:
# CELL 18: INTEGRATED GRADIENTS EXPLAINER


In [ ]:
class IntegratedGradientsExplainer:
    """Integrated Gradients explainer using Captum"""
    
    def __init__(self, model):
        self.model = model
        self.ig = IntegratedGradients(model)
        
    def explain(self, input_tensor, target_class=None, n_steps=50):
        """
        Generate Integrated Gradients attribution
        
        Args:
            input_tensor: Input image tensor (1, C, H, W)
            target_class: Target class index
            n_steps: Number of integration steps
        
        Returns:
            heatmap: Normalized attribution map (H, W)
        """
        if target_class is None:
            with torch.no_grad():
                output = self.model(input_tensor)
                target_class = output.argmax(dim=1).item()
        
        # Create baseline (black image)
        baseline = torch.zeros_like(input_tensor).to(input_tensor.device)
        
        # Compute attributions
        attributions = self.ig.attribute(
            input_tensor, 
            baselines=baseline, 
            target=target_class,
            n_steps=n_steps
        )
        
        # Convert to heatmap (sum across channels, take absolute value)
        heatmap = attributions.squeeze().abs().sum(dim=0).cpu().numpy()
        
        return normalize_heatmap(heatmap)
    
    def visualize(self, input_tensor, heatmap):
        """Overlay heatmap on original image"""
        rgb_img = tensor_to_numpy(input_tensor)
        visualization = show_cam_on_image(rgb_img, heatmap, use_rgb=True)
        return visualization


In [ ]:
# CELL 19: SHAP (GradientSHAP) EXPLAINER


In [ ]:
class SHAPExplainer:
    """SHAP explainer using GradientSHAP from Captum"""
    
    def __init__(self, model, background_data=None):
        self.model = model
        self.gradient_shap = GradientShap(model)
        self.background_data = background_data
        
    def set_background(self, data_loader, n_samples=50):
        """Set background data for SHAP"""
        background_images = []
        count = 0
        
        for images, _, _ in data_loader:
            for img in images:
                background_images.append(img)
                count += 1
                if count >= n_samples:
                    break
            if count >= n_samples:
                break
        
        self.background_data = torch.stack(background_images).to(device)
        print(f"Background data set: {self.background_data.shape}")
        
    def explain(self, input_tensor, target_class=None, n_samples=25):
        """
        Generate GradientSHAP attribution
        
        Args:
            input_tensor: Input image tensor (1, C, H, W)
            target_class: Target class index
            n_samples: Number of samples for approximation
        
        Returns:
            heatmap: Normalized attribution map (H, W)
        """
        if target_class is None:
            with torch.no_grad():
                output = self.model(input_tensor)
                target_class = output.argmax(dim=1).item()
        
        # Use random baselines if no background data
        if self.background_data is None:
            baselines = torch.randn(n_samples, *input_tensor.shape[1:]).to(input_tensor.device) * 0.001
        else:
            # Sample from background data
            indices = torch.randperm(self.background_data.size(0))[:n_samples]
            baselines = self.background_data[indices]
        
        # Compute attributions
        attributions = self.gradient_shap.attribute(
            input_tensor,
            baselines=baselines,
            target=target_class,
            n_samples=n_samples
        )
        
        # Convert to heatmap
        heatmap = attributions.squeeze().abs().sum(dim=0).cpu().numpy()
        
        return normalize_heatmap(heatmap)
    
    def visualize(self, input_tensor, heatmap):
        """Overlay heatmap on original image"""
        rgb_img = tensor_to_numpy(input_tensor)
        visualization = show_cam_on_image(rgb_img, heatmap, use_rgb=True)
        return visualization


In [ ]:
# CELL 20: LIME EXPLAINER


In [ ]:
class LIMEExplainer:
    """LIME explainer for image classification"""
    
    def __init__(self, model, device):
        self.model = model
        self.device = device
        self.explainer = lime_image.LimeImageExplainer()
        
    def _batch_predict(self, images):
        """Prediction function for LIME"""
        self.model.eval()
        
        # Convert numpy arrays to tensors
        batch = []
        for img in images:
            img = img.astype(np.float32) / 255.0
            img = (img - np.array([0.485, 0.456, 0.406])) / np.array([0.229, 0.224, 0.225])
            img = torch.tensor(img).permute(2, 0, 1).float()
            batch.append(img)
        
        batch = torch.stack(batch).to(self.device)
        
        with torch.no_grad():
            outputs = self.model(batch)
            probs = F.softmax(outputs, dim=1).cpu().numpy()
        
        return probs
    
    def explain(self, input_tensor, target_class=None, num_samples=500, num_features=10):
        """
        Generate LIME explanation
        
        Args:
            input_tensor: Input image tensor (1, C, H, W)
            target_class: Target class index
            num_samples: Number of perturbed samples
            num_features: Number of superpixels to include
        
        Returns:
            heatmap: Normalized attribution map (H, W)
        """
        # Convert tensor to numpy image (0-255)
        rgb_img = tensor_to_numpy(input_tensor)
        rgb_img_uint8 = (rgb_img * 255).astype(np.uint8)
        
        if target_class is None:
            with torch.no_grad():
                output = self.model(input_tensor)
                target_class = output.argmax(dim=1).item()
        
        # Generate explanation
        explanation = self.explainer.explain_instance(
            rgb_img_uint8,
            self._batch_predict,
            top_labels=2,
            hide_color=0,
            num_samples=num_samples
        )
        
        # Get mask for target class
        temp, mask = explanation.get_image_and_mask(
            target_class,
            positive_only=True,
            num_features=num_features,
            hide_rest=False
        )
        
        # Create heatmap from mask
        heatmap = np.zeros_like(mask, dtype=np.float32)
        
        local_exp = explanation.local_exp[target_class]
        segments = explanation.segments
        
        for segment_idx, weight in local_exp:
            if weight > 0:
                heatmap[segments == segment_idx] = weight
        
        return normalize_heatmap(heatmap)
    
    def visualize(self, input_tensor, heatmap):
        """Overlay heatmap on original image"""
        rgb_img = tensor_to_numpy(input_tensor)
        visualization = show_cam_on_image(rgb_img, heatmap, use_rgb=True)
        return visualization


In [ ]:
# CELL 21: INITIALIZE ALL EXPLAINERS


In [ ]:
# Ensure model is in eval mode
model.eval()

# Get target layer for Grad-CAM
target_layer = model.get_target_layer()

# Initialize explainers
print("Initializing XAI explainers...")

gradcam_explainer = GradCAMExplainer(model, target_layer)
print("✓ Grad-CAM++ explainer initialized")

ig_explainer = IntegratedGradientsExplainer(model)
print("✓ Integrated Gradients explainer initialized")

shap_explainer = SHAPExplainer(model)
shap_explainer.set_background(train_loader, n_samples=30)
print("✓ SHAP explainer initialized")

lime_explainer = LIMEExplainer(model, device)
print("✓ LIME explainer initialized")

# Create explainers dictionary
explainers = {
    'gradcam': gradcam_explainer,
    'ig': ig_explainer,
    'shap': shap_explainer,
    'lime': lime_explainer
}

print("\nAll XAI explainers ready!")


In [ ]:
# CELL 22: GENERATE EXPLANATIONS FUNCTION


In [ ]:
def generate_all_explanations(model, image_tensor, explainers, target_class=1):
    """
    Generate explanations using all XAI methods
    
    Args:
        model: The trained model
        image_tensor: Input image tensor (1, C, H, W)
        explainers: Dictionary of explainer objects
        target_class: Target class for explanation (1 = cancer)
    
    Returns:
        Dictionary of heatmaps for each method
    """
    heatmaps = {}
    
    print("  Generating Grad-CAM++ explanation...")
    heatmaps['gradcam++'] = explainers['gradcam'].explain(image_tensor, target_class)
    
    print("  Generating Integrated Gradients explanation...")
    heatmaps['integrated_gradients'] = explainers['ig'].explain(image_tensor, target_class)
    
    print("  Generating SHAP explanation...")
    heatmaps['shap'] = explainers['shap'].explain(image_tensor, target_class)
    
    print("  Generating LIME explanation...")
    heatmaps['lime'] = explainers['lime'].explain(image_tensor, target_class)
    
    return heatmaps


In [ ]:
# CELL 23: VISUALIZE XAI COMPARISON


In [ ]:
def visualize_xai_comparison(test_loader, model, explainers, num_samples=4, save_path=None):
    """Visualize XAI method comparisons on sample images"""
    
    model.eval()
    
    # Collect samples from each class
    cancer_samples = []
    non_cancer_samples = []
    
    for images, labels, paths in test_loader:
        for i in range(len(labels)):
            if labels[i] == 1 and len(cancer_samples) < num_samples // 2:
                cancer_samples.append((images[i:i+1].to(device), labels[i].item(), paths[i]))
            elif labels[i] == 0 and len(non_cancer_samples) < num_samples // 2:
                non_cancer_samples.append((images[i:i+1].to(device), labels[i].item(), paths[i]))
        
        if len(cancer_samples) >= num_samples // 2 and len(non_cancer_samples) >= num_samples // 2:
            break
    
    samples = cancer_samples + non_cancer_samples
    
    if len(samples) == 0:
        print("No samples found!")
        return None, None
    
    # Create figure
    fig, axes = plt.subplots(len(samples), 6, figsize=(24, 4 * len(samples)))
    
    if len(samples) == 1:
        axes = axes.reshape(1, -1)
    
    method_names = ['Original', 'Grad-CAM++', 'Integrated\nGradients', 'SHAP', 'LIME', 'Prediction']
    
    all_heatmaps = []
    
    for idx, (image_tensor, label, path) in enumerate(samples):
        print(f"\nProcessing sample {idx + 1}/{len(samples)}: {os.path.basename(path)}")
        
        # Get prediction
        with torch.no_grad():
            output = model(image_tensor)
            probs = F.softmax(output, dim=1)
            pred_class = output.argmax(dim=1).item()
            confidence = probs[0, pred_class].item()
        
        # Generate explanations
        heatmaps = generate_all_explanations(model, image_tensor, explainers, target_class=1)
        all_heatmaps.append(heatmaps)
        
        # Original image
        rgb_img = tensor_to_numpy(image_tensor)
        axes[idx, 0].imshow(rgb_img)
        label_text = 'Cancer' if label == 1 else 'Non-Cancer'
        axes[idx, 0].set_title(f'{method_names[0]}\n(True: {label_text})', fontsize=10)
        axes[idx, 0].axis('off')
        
        # XAI visualizations
        xai_methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
        for j, method in enumerate(xai_methods):
            vis = show_cam_on_image(rgb_img, heatmaps[method], use_rgb=True)
            axes[idx, j + 1].imshow(vis)
            axes[idx, j + 1].set_title(method_names[j + 1], fontsize=10)
            axes[idx, j + 1].axis('off')
        
        # Prediction info
        axes[idx, 5].axis('off')
        pred_text = 'Cancer' if pred_class == 1 else 'Non-Cancer'
        color = 'green' if pred_class == label else 'red'
        status = "✓ Correct" if pred_class == label else "✗ Wrong"
        axes[idx, 5].text(0.5, 0.5, 
                         f'Pred: {pred_text}\n\nConf: {confidence*100:.1f}%\n\n{status}',
                         ha='center', va='center', fontsize=14, color=color,
                         transform=axes[idx, 5].transAxes)
    
    plt.suptitle('Multi-Method XAI Comparison for Colon Cancer Classification', 
                 fontsize=16, fontweight='bold', y=1.02)
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    return samples, all_heatmaps


# Generate XAI comparison visualization
print("Generating XAI comparison visualizations...")
samples, sample_heatmaps = visualize_xai_comparison(
    test_loader, model, explainers, num_samples=4,
    save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'xai_comparison.png')
)

print("\n" + "="*50)
print("PART 3 COMPLETE: XAI Methods Implementation")
print("="*50)


In [ ]:
"""
============================================================================
PART 4: QUANTITATIVE XAI EVALUATION
============================================================================
"""


In [ ]:
# CELL 24: FAITHFULNESS EVALUATOR


In [ ]:
class FaithfulnessEvaluator:
    """Evaluate faithfulness of XAI explanations using deletion/insertion tests"""
    
    def __init__(self, model, device, steps=20):
        self.model = model
        self.device = device
        self.steps = steps
        
    def _get_baseline(self, image, method='blur'):
        """Get baseline image for insertion test"""
        if method == 'blur':
            img_np = image.squeeze().permute(1, 2, 0).cpu().numpy()
            blurred = cv2.GaussianBlur(img_np, (51, 51), 20)
            return torch.tensor(blurred).permute(2, 0, 1).unsqueeze(0).float().to(self.device)
        else:
            return torch.full_like(image, image.mean())
    
    def deletion_test(self, image, heatmap, target_class):
        """
        Deletion test: Remove most important pixels progressively
        
        Lower AUC = better explanation (probability drops quickly when important pixels removed)
        """
        self.model.eval()
        
        h, w = heatmap.shape
        flat_heatmap = heatmap.flatten()
        sorted_indices = np.argsort(-flat_heatmap)  # Descending order
        
        baseline_value = image.mean()
        scores = []
        num_pixels = len(flat_heatmap)
        
        for step in range(self.steps + 1):
            k = int((step / self.steps) * num_pixels)
            
            mask = np.ones(num_pixels)
            if k > 0:
                mask[sorted_indices[:k]] = 0
            mask = mask.reshape(h, w)
            mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(self.device)
            mask_tensor = mask_tensor.expand_as(image)
            
            masked_image = image * mask_tensor + baseline_value * (1 - mask_tensor)
            
            with torch.no_grad():
                output = self.model(masked_image)
                prob = F.softmax(output, dim=1)[0, target_class].item()
            
            scores.append(prob)
        
        auc = np.trapz(scores) / len(scores)
        return scores, auc
    
    def insertion_test(self, image, heatmap, target_class):
        """
        Insertion test: Start from baseline and insert most important pixels progressively
        
        Higher AUC = better explanation (probability rises quickly when important pixels added)
        """
        self.model.eval()
        
        baseline = self._get_baseline(image)
        
        h, w = heatmap.shape
        flat_heatmap = heatmap.flatten()
        sorted_indices = np.argsort(-flat_heatmap)
        
        scores = []
        num_pixels = len(flat_heatmap)
        
        for step in range(self.steps + 1):
            k = int((step / self.steps) * num_pixels)
            
            mask = np.zeros(num_pixels)
            if k > 0:
                mask[sorted_indices[:k]] = 1
            mask = mask.reshape(h, w)
            mask_tensor = torch.tensor(mask, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(self.device)
            mask_tensor = mask_tensor.expand_as(image)
            
            inserted_image = baseline * (1 - mask_tensor) + image * mask_tensor
            
            with torch.no_grad():
                output = self.model(inserted_image)
                prob = F.softmax(output, dim=1)[0, target_class].item()
            
            scores.append(prob)
        
        auc = np.trapz(scores) / len(scores)
        return scores, auc


In [ ]:
# CELL 25: RUN FAITHFULNESS TESTS


In [ ]:
def run_faithfulness_evaluation(test_loader, model, explainers, evaluator, num_samples=20):
    """Run faithfulness tests on multiple samples"""
    
    results = {method: {'deletion_scores': [], 'insertion_scores': [], 
                        'deletion_aucs': [], 'insertion_aucs': []} 
               for method in ['gradcam++', 'integrated_gradients', 'shap', 'lime']}
    
    model.eval()
    sample_count = 0
    
    for images, labels, paths in test_loader:
        for i in range(len(labels)):
            if sample_count >= num_samples:
                break
                
            image = images[i:i+1].to(device)
            label = labels[i].item()
            
            # Only test on correctly classified cancer samples
            with torch.no_grad():
                output = model(image)
                pred = output.argmax(dim=1).item()
            
            if pred != 1 or label != 1:  # Skip if not cancer or misclassified
                continue
            
            print(f"\nProcessing sample {sample_count + 1}/{num_samples}")
            
            # Generate explanations and run tests for each method
            for method_name, method_key in [('gradcam++', 'gradcam'), 
                                            ('integrated_gradients', 'ig'),
                                            ('shap', 'shap'), 
                                            ('lime', 'lime')]:
                print(f"  Testing {method_name}...")
                
                # Generate heatmap
                if method_key == 'gradcam':
                    heatmap = explainers[method_key].explain(image, target_class=1)
                elif method_key == 'ig':
                    heatmap = explainers[method_key].explain(image, target_class=1, n_steps=30)
                elif method_key == 'shap':
                    heatmap = explainers[method_key].explain(image, target_class=1, n_samples=15)
                else:  # lime
                    heatmap = explainers[method_key].explain(image, target_class=1, num_samples=300)
                
                # Run deletion test
                del_scores, del_auc = evaluator.deletion_test(image, heatmap, target_class=1)
                results[method_name]['deletion_scores'].append(del_scores)
                results[method_name]['deletion_aucs'].append(del_auc)
                
                # Run insertion test
                ins_scores, ins_auc = evaluator.insertion_test(image, heatmap, target_class=1)
                results[method_name]['insertion_scores'].append(ins_scores)
                results[method_name]['insertion_aucs'].append(ins_auc)
            
            sample_count += 1
        
        if sample_count >= num_samples:
            break
    
    return results


# Initialize evaluator and run tests
print("\nRunning Faithfulness Evaluation...")
print("="*50)

faithfulness_evaluator = FaithfulnessEvaluator(model, device, steps=20)
faithfulness_results = run_faithfulness_evaluation(
    test_loader, model, explainers, faithfulness_evaluator, num_samples=15
)


In [ ]:
# CELL 26: PLOT FAITHFULNESS RESULTS


In [ ]:
def plot_faithfulness_results(results, save_path=None):
    """Plot deletion/insertion curves and AUC comparison"""
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    
    methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
    colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
    method_labels = ['Grad-CAM++', 'Integrated Gradients', 'SHAP', 'LIME']
    
    steps = np.linspace(0, 100, len(results[methods[0]]['deletion_scores'][0]))
    
    # Deletion curves
    ax = axes[0, 0]
    for method, color, label in zip(methods, colors, method_labels):
        scores = np.array(results[method]['deletion_scores'])
        mean_scores = scores.mean(axis=0)
        std_scores = scores.std(axis=0)
        
        ax.plot(steps, mean_scores, color=color, linewidth=2, label=label)
        ax.fill_between(steps, mean_scores - std_scores, mean_scores + std_scores, 
                       color=color, alpha=0.2)
    
    ax.set_xlabel('Pixels Removed (%)', fontsize=12)
    ax.set_ylabel('Cancer Probability', fontsize=12)
    ax.set_title('Deletion Test (Lower AUC = Better)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Insertion curves
    ax = axes[0, 1]
    for method, color, label in zip(methods, colors, method_labels):
        scores = np.array(results[method]['insertion_scores'])
        mean_scores = scores.mean(axis=0)
        std_scores = scores.std(axis=0)
        
        ax.plot(steps, mean_scores, color=color, linewidth=2, label=label)
        ax.fill_between(steps, mean_scores - std_scores, mean_scores + std_scores, 
                       color=color, alpha=0.2)
    
    ax.set_xlabel('Pixels Inserted (%)', fontsize=12)
    ax.set_ylabel('Cancer Probability', fontsize=12)
    ax.set_title('Insertion Test (Higher AUC = Better)', fontsize=14, fontweight='bold')
    ax.legend(fontsize=10)
    ax.grid(True, alpha=0.3)
    
    # Deletion AUC comparison
    ax = axes[1, 0]
    deletion_aucs = [results[m]['deletion_aucs'] for m in methods]
    bp1 = ax.boxplot(deletion_aucs, labels=method_labels, patch_artist=True)
    for patch, color in zip(bp1['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel('Deletion AUC', fontsize=12)
    ax.set_title('Deletion AUC Distribution (Lower = Better)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    # Insertion AUC comparison
    ax = axes[1, 1]
    insertion_aucs = [results[m]['insertion_aucs'] for m in methods]
    bp2 = ax.boxplot(insertion_aucs, labels=method_labels, patch_artist=True)
    for patch, color in zip(bp2['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel('Insertion AUC', fontsize=12)
    ax.set_title('Insertion AUC Distribution (Higher = Better)', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print summary statistics
    print("\n" + "="*60)
    print("FAITHFULNESS TEST SUMMARY")
    print("="*60)
    print(f"{'Method':<25} {'Del AUC (↓)':<15} {'Ins AUC (↑)':<15}")
    print("-"*55)
    for method, label in zip(methods, method_labels):
        del_mean = np.mean(results[method]['deletion_aucs'])
        del_std = np.std(results[method]['deletion_aucs'])
        ins_mean = np.mean(results[method]['insertion_aucs'])
        ins_std = np.std(results[method]['insertion_aucs'])
        print(f"{label:<25} {del_mean:.3f}±{del_std:.3f}     {ins_mean:.3f}±{ins_std:.3f}")
    print("="*60)


plot_faithfulness_results(faithfulness_results, 
                         save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'faithfulness_results.png'))


In [ ]:
# CELL 27: SANITY CHECK - MODEL RANDOMIZATION


In [ ]:
def sanity_check_randomization(model, test_loader, explainers, num_samples=5):
    """
    Sanity check: Compare explanations from trained vs randomized model
    
    If explanations are similar after randomization, the XAI method might be unreliable
    """
    
    print("\n" + "="*60)
    print("SANITY CHECK: MODEL RANDOMIZATION TEST")
    print("="*60)
    
    # Get a sample image
    for images, labels, paths in test_loader:
        # Find a cancer sample
        for i in range(len(labels)):
            if labels[i] == 1:
                sample_image = images[i:i+1].to(device)
                sample_path = paths[i]
                break
        break
    
    print(f"Testing on: {os.path.basename(sample_path)}")
    
    # Get explanations from trained model
    model.eval()
    trained_heatmaps = {}
    
    print("\nGenerating explanations from trained model...")
    trained_heatmaps['gradcam++'] = explainers['gradcam'].explain(sample_image, target_class=1)
    trained_heatmaps['integrated_gradients'] = explainers['ig'].explain(sample_image, target_class=1)
    
    # Create randomized model
    print("Creating randomized model...")
    random_model = ColonCancerClassifier(
        model_name=Config.MODEL_NAME,
        num_classes=Config.NUM_CLASSES,
        pretrained=False
    ).to(device)
    
    # Reinitialize with random weights
    for m in random_model.modules():
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    random_model.eval()
    
    # Create explainers for random model
    random_target_layer = random_model.get_target_layer()
    random_gradcam = GradCAMExplainer(random_model, random_target_layer)
    random_ig = IntegratedGradientsExplainer(random_model)
    
    # Get explanations from randomized model
    print("Generating explanations from randomized model...")
    random_heatmaps = {}
    random_heatmaps['gradcam++'] = random_gradcam.explain(sample_image, target_class=1)
    random_heatmaps['integrated_gradients'] = random_ig.explain(sample_image, target_class=1)
    
    # Visualize comparison
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    rgb_img = tensor_to_numpy(sample_image)
    
    # Original image
    axes[0, 0].imshow(rgb_img)
    axes[0, 0].set_title('Original Image', fontsize=12)
    axes[0, 0].axis('off')
    
    # Trained model - Grad-CAM++
    vis_trained_gc = show_cam_on_image(rgb_img, trained_heatmaps['gradcam++'], use_rgb=True)
    axes[0, 1].imshow(vis_trained_gc)
    axes[0, 1].set_title('Trained Model\nGrad-CAM++', fontsize=12)
    axes[0, 1].axis('off')
    
    # Trained model - IG
    vis_trained_ig = show_cam_on_image(rgb_img, trained_heatmaps['integrated_gradients'], use_rgb=True)
    axes[0, 2].imshow(vis_trained_ig)
    axes[0, 2].set_title('Trained Model\nIntegrated Gradients', fontsize=12)
    axes[0, 2].axis('off')
    
    # Spacer
    axes[1, 0].axis('off')
    
    # Random model - Grad-CAM++
    vis_random_gc = show_cam_on_image(rgb_img, random_heatmaps['gradcam++'], use_rgb=True)
    axes[1, 1].imshow(vis_random_gc)
    axes[1, 1].set_title('Random Model\nGrad-CAM++', fontsize=12)
    axes[1, 1].axis('off')
    
    # Random model - IG
    vis_random_ig = show_cam_on_image(rgb_img, random_heatmaps['integrated_gradients'], use_rgb=True)
    axes[1, 2].imshow(vis_random_ig)
    axes[1, 2].set_title('Random Model\nIntegrated Gradients', fontsize=12)
    axes[1, 2].axis('off')
    
    plt.suptitle('Sanity Check: Trained vs Randomized Model Explanations', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'sanity_check.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Calculate correlation between heatmaps
    print("\nCorrelation Analysis:")
    for method in ['gradcam++', 'integrated_gradients']:
        corr = np.corrcoef(trained_heatmaps[method].flatten(), 
                          random_heatmaps[method].flatten())[0, 1]
        print(f"  {method}: Correlation = {corr:.4f}")
        if abs(corr) < 0.3:
            print(f"    ✓ Low correlation - explanation changes appropriately with model")
        else:
            print(f"    ⚠ High correlation - may indicate issues with explanation method")


# Run sanity check
sanity_check_randomization(model, test_loader, explainers)

print("\n" + "="*50)
print("PART 4 COMPLETE: Quantitative XAI Evaluation")
print("="*50)


In [ ]:
#re run this code for the above cell erro 207 cell
def sanity_check_randomization(model, test_loader, explainers, num_samples=5):
    """
    Sanity check: Compare explanations from trained vs randomized model
    """
    
    print("\n" + "="*60)
    print("SANITY CHECK: MODEL RANDOMIZATION TEST")
    print("="*60)
    
    # Get a sample image - FIXED VERSION
    sample_image = None
    sample_path = None
    
    for images, labels, paths in test_loader:
        for i in range(len(labels)):
            if labels[i] == 1:  # Find a cancer sample
                sample_image = images[i:i+1].to(device)
                sample_path = paths[i]
                break
        if sample_image is not None:
            break
    
    # If no cancer sample found, use any sample
    if sample_image is None:
        for images, labels, paths in test_loader:
            sample_image = images[0:1].to(device)
            sample_path = paths[0]
            break
    
    print(f"Testing on: {os.path.basename(sample_path)}")
    
    # Get explanations from trained model
    model.eval()
    trained_heatmaps = {}
    
    print("\nGenerating explanations from trained model...")
    trained_heatmaps['gradcam++'] = explainers['gradcam'].explain(sample_image, target_class=1)
    trained_heatmaps['integrated_gradients'] = explainers['ig'].explain(sample_image, target_class=1)
    
    # Create randomized model
    print("Creating randomized model...")
    random_model = ColonCancerClassifier(
        model_name=Config.MODEL_NAME,
        num_classes=Config.NUM_CLASSES,
        pretrained=False
    ).to(device)
    
    # Reinitialize with random weights
    for m in random_model.modules():
        if isinstance(m, nn.Conv2d) or isinstance(m, nn.Linear):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
            if m.bias is not None:
                nn.init.constant_(m.bias, 0)
    
    random_model.eval()
    
    # Create explainers for random model
    random_target_layer = random_model.get_target_layer()
    random_gradcam = GradCAMExplainer(random_model, random_target_layer)
    random_ig = IntegratedGradientsExplainer(random_model)
    
    # Get explanations from randomized model
    print("Generating explanations from randomized model...")
    random_heatmaps = {}
    random_heatmaps['gradcam++'] = random_gradcam.explain(sample_image, target_class=1)
    random_heatmaps['integrated_gradients'] = random_ig.explain(sample_image, target_class=1)
    
    # Visualize comparison
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    
    rgb_img = tensor_to_numpy(sample_image)
    
    # Original image
    axes[0, 0].imshow(rgb_img)
    axes[0, 0].set_title('Original Image', fontsize=12)
    axes[0, 0].axis('off')
    
    # Trained model - Grad-CAM++
    vis_trained_gc = show_cam_on_image(rgb_img, trained_heatmaps['gradcam++'], use_rgb=True)
    axes[0, 1].imshow(vis_trained_gc)
    axes[0, 1].set_title('Trained Model\nGrad-CAM++', fontsize=12)
    axes[0, 1].axis('off')
    
    # Trained model - IG
    vis_trained_ig = show_cam_on_image(rgb_img, trained_heatmaps['integrated_gradients'], use_rgb=True)
    axes[0, 2].imshow(vis_trained_ig)
    axes[0, 2].set_title('Trained Model\nIntegrated Gradients', fontsize=12)
    axes[0, 2].axis('off')
    
    # Spacer
    axes[1, 0].axis('off')
    
    # Random model - Grad-CAM++
    vis_random_gc = show_cam_on_image(rgb_img, random_heatmaps['gradcam++'], use_rgb=True)
    axes[1, 1].imshow(vis_random_gc)
    axes[1, 1].set_title('Random Model\nGrad-CAM++', fontsize=12)
    axes[1, 1].axis('off')
    
    # Random model - IG
    vis_random_ig = show_cam_on_image(rgb_img, random_heatmaps['integrated_gradients'], use_rgb=True)
    axes[1, 2].imshow(vis_random_ig)
    axes[1, 2].set_title('Random Model\nIntegrated Gradients', fontsize=12)
    axes[1, 2].axis('off')
    
    plt.suptitle('Sanity Check: Trained vs Randomized Model Explanations', 
                fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'sanity_check.png'), dpi=150, bbox_inches='tight')
    plt.show()
    
    # Calculate correlation between heatmaps
    print("\nCorrelation Analysis:")
    for method in ['gradcam++', 'integrated_gradients']:
        corr = np.corrcoef(trained_heatmaps[method].flatten(), 
                          random_heatmaps[method].flatten())[0, 1]
        print(f"  {method}: Correlation = {corr:.4f}")
        if abs(corr) < 0.3:
            print(f"    ✓ Low correlation - explanation changes appropriately with model")
        else:
            print(f"    ⚠ High correlation - may indicate issues with explanation method")


# Run sanity check
sanity_check_randomization(model, test_loader, explainers)

print("\n" + "="*50)
print("PART 4 COMPLETE: Quantitative XAI Evaluation")
print("="*50)

In [ ]:
"""
============================================================================
PART 5A: EXPLANATION-GUIDED FOCAL RADIOMICS - ROI EXTRACTION
============================================================================
"""


In [ ]:
# CELL 28: ROI EXTRACTION FROM HEATMAPS


In [ ]:
class FocalROIExtractor:
    """Extract focal regions of interest from XAI heatmaps"""
    
    def __init__(self, threshold_percentile=85):
        self.threshold_percentile = threshold_percentile
    
    def extract_roi(self, heatmap, min_area=100):
        """
        Extract ROI mask from heatmap by thresholding
        
        Args:
            heatmap: Normalized heatmap (H, W)
            min_area: Minimum area for valid ROI
        
        Returns:
            roi_mask: Binary mask of the focal region
            bbox: Bounding box (x, y, w, h)
        """
        threshold = np.percentile(heatmap, self.threshold_percentile)
        binary_mask = (heatmap >= threshold).astype(np.uint8)
        
        labeled, num_features = scipy_label(binary_mask)
        
        if num_features == 0:
            return np.ones_like(heatmap, dtype=np.uint8), (0, 0, heatmap.shape[1], heatmap.shape[0])
        
        largest_cc = None
        largest_area = 0
        
        for i in range(1, num_features + 1):
            component = (labeled == i).astype(np.uint8)
            area = component.sum()
            if area > largest_area:
                largest_area = area
                largest_cc = component
        
        if largest_cc is None or largest_area < min_area:
            return np.ones_like(heatmap, dtype=np.uint8), (0, 0, heatmap.shape[1], heatmap.shape[0])
        
        kernel = np.ones((5, 5), np.uint8)
        roi_mask = cv2.morphologyEx(largest_cc, cv2.MORPH_CLOSE, kernel)
        roi_mask = cv2.morphologyEx(roi_mask, cv2.MORPH_OPEN, kernel)
        
        coords = np.where(roi_mask > 0)
        if len(coords[0]) == 0:
            return np.ones_like(heatmap, dtype=np.uint8), (0, 0, heatmap.shape[1], heatmap.shape[0])
        
        y_min, y_max = coords[0].min(), coords[0].max()
        x_min, x_max = coords[1].min(), coords[1].max()
        bbox = (x_min, y_min, x_max - x_min, y_max - y_min)
        
        return roi_mask, bbox


# Initialize ROI extractor
roi_extractor = FocalROIExtractor(threshold_percentile=85)
print("ROI Extractor initialized.")


In [ ]:
# CELL 29: RADIOMICS FEATURE EXTRACTION


In [ ]:
class RadiomicsExtractor:
    """Extract radiomics features from images and ROIs"""
    
    def __init__(self, bin_width=25):
        self.bin_width = bin_width
        
    def extract_features(self, image, mask):
        """Extract custom radiomics features"""
        
        if len(image.shape) == 3:
            gray = cv2.cvtColor((image * 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        else:
            gray = (image * 255).astype(np.uint8)
        
        if mask.sum() > 0:
            roi_pixels = gray[mask > 0]
        else:
            roi_pixels = gray.flatten()
        
        features = {}
        
        # First-order statistics
        features['mean'] = np.mean(roi_pixels)
        features['std'] = np.std(roi_pixels)
        features['min'] = np.min(roi_pixels)
        features['max'] = np.max(roi_pixels)
        features['median'] = np.median(roi_pixels)
        features['skewness'] = stats.skew(roi_pixels) if len(roi_pixels) > 2 else 0
        features['kurtosis'] = stats.kurtosis(roi_pixels) if len(roi_pixels) > 2 else 0
        features['entropy'] = stats.entropy(np.histogram(roi_pixels, bins=64)[0] + 1e-10)
        features['energy'] = np.sum(roi_pixels.astype(np.float64) ** 2) / len(roi_pixels)
        features['range'] = features['max'] - features['min']
        
        # Percentiles
        features['percentile_10'] = np.percentile(roi_pixels, 10)
        features['percentile_25'] = np.percentile(roi_pixels, 25)
        features['percentile_75'] = np.percentile(roi_pixels, 75)
        features['percentile_90'] = np.percentile(roi_pixels, 90)
        features['iqr'] = features['percentile_75'] - features['percentile_25']
        
        # Shape features
        if mask.sum() > 10:
            features['area'] = mask.sum()
            contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            if len(contours) > 0:
                features['perimeter'] = cv2.arcLength(contours[0], True)
                features['compactness'] = (4 * np.pi * features['area']) / (features['perimeter'] ** 2 + 1e-10)
            else:
                features['perimeter'] = 0
                features['compactness'] = 0
        else:
            features['area'] = gray.shape[0] * gray.shape[1]
            features['perimeter'] = 2 * (gray.shape[0] + gray.shape[1])
            features['compactness'] = 1.0
        
        # GLCM features (simplified)
        try:
            from skimage.feature import graycomatrix, graycoprops
            glcm = graycomatrix(gray, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4], 
                               levels=256, symmetric=True, normed=True)
            features['glcm_contrast'] = graycoprops(glcm, 'contrast').mean()
            features['glcm_dissimilarity'] = graycoprops(glcm, 'dissimilarity').mean()
            features['glcm_homogeneity'] = graycoprops(glcm, 'homogeneity').mean()
            features['glcm_energy'] = graycoprops(glcm, 'energy').mean()
            features['glcm_correlation'] = graycoprops(glcm, 'correlation').mean()
        except:
            features['glcm_contrast'] = 0
            features['glcm_dissimilarity'] = 0
            features['glcm_homogeneity'] = 0
            features['glcm_energy'] = 0
            features['glcm_correlation'] = 0
        
        return features


# Initialize radiomics extractor
radiomics_extractor = RadiomicsExtractor(bin_width=Config.RADIOMICS_BIN_WIDTH)
print("Radiomics Extractor initialized.")


In [ ]:
"""
============================================================================
PART 5B: RADIOMICS EXTRACTION PIPELINE AND FUSION MODEL
============================================================================
"""


In [ ]:
# CELL 30: EXTRACT ALL RADIOMICS FEATURES


In [ ]:
def extract_all_radiomics(data_loader, model, explainers, roi_extractor, radiomics_extractor, 
                          xai_method='gradcam++', max_samples=None):
    """
    Extract radiomics features for all samples
    """
    model.eval()
    
    whole_image_features = []
    xai_roi_features = []
    deep_features = []
    labels = []
    
    sample_count = 0
    
    print(f"\nExtracting features using {xai_method}...")
    
    for images, batch_labels, paths in tqdm(data_loader, desc="Extracting"):
        for i in range(len(batch_labels)):
            if max_samples and sample_count >= max_samples:
                break
            
            image = images[i:i+1].to(device)
            label = batch_labels[i].item()
            
            rgb_img = tensor_to_numpy(image)
            
            # Get XAI heatmap
            if xai_method == 'gradcam++':
                heatmap = explainers['gradcam'].explain(image, target_class=1)
            elif xai_method == 'integrated_gradients':
                heatmap = explainers['ig'].explain(image, target_class=1, n_steps=30)
            elif xai_method == 'shap':
                heatmap = explainers['shap'].explain(image, target_class=1, n_samples=15)
            else:
                heatmap = explainers['lime'].explain(image, target_class=1, num_samples=300)
            
            # Extract ROI
            roi_mask, bbox = roi_extractor.extract_roi(heatmap)
            
            # Extract whole image radiomics
            whole_mask = np.ones((Config.IMG_SIZE, Config.IMG_SIZE), dtype=np.uint8)
            whole_feat = radiomics_extractor.extract_features(rgb_img, whole_mask)
            
            # Extract XAI ROI radiomics
            xai_feat = radiomics_extractor.extract_features(rgb_img, roi_mask)
            
            # Extract deep features
            with torch.no_grad():
                deep_feat = model.get_features(image).cpu().numpy().flatten()
            
            whole_image_features.append(whole_feat)
            xai_roi_features.append(xai_feat)
            deep_features.append(deep_feat)
            labels.append(label)
            
            sample_count += 1
        
        if max_samples and sample_count >= max_samples:
            break
    
    # Convert to DataFrames
    whole_df = pd.DataFrame(whole_image_features)
    xai_df = pd.DataFrame(xai_roi_features)
    deep_df = pd.DataFrame(deep_features)
    
    # Add prefix to distinguish features
    whole_df.columns = ['whole_' + str(c) for c in whole_df.columns]
    xai_df.columns = ['xai_' + str(c) for c in xai_df.columns]
    deep_df.columns = ['deep_' + str(i) for i in range(deep_df.shape[1])]
    
    return whole_df, xai_df, deep_df, np.array(labels)


# Extract features
print("\nExtracting radiomics features from training set...")
train_whole, train_xai, train_deep, train_labels_rad = extract_all_radiomics(
    train_loader, model, explainers, roi_extractor, radiomics_extractor,
    xai_method='gradcam++', max_samples=None
)

print("\nExtracting radiomics features from test set...")
test_whole, test_xai, test_deep, test_labels_rad = extract_all_radiomics(
    test_loader, model, explainers, roi_extractor, radiomics_extractor,
    xai_method='gradcam++', max_samples=None
)

print(f"\nFeature shapes:")
print(f"  Whole image radiomics: {train_whole.shape}")
print(f"  XAI ROI radiomics: {train_xai.shape}")
print(f"  Deep features: {train_deep.shape}")


In [ ]:
# Check if variables exist
print(f"train_whole shape: {train_whole.shape}")
print(f"train_xai shape: {train_xai.shape}")
print(f"train_deep shape: {train_deep.shape}")
print(f"test_whole shape: {test_whole.shape}")

In [ ]:
# ===========================================
# SAVE EXTRACTED FEATURES TO EXCEL
# ===========================================

def save_features_to_excel(train_whole, train_xai, train_deep, train_labels,
                           test_whole, test_xai, test_deep, test_labels,
                           save_path):
    """Save all extracted features to Excel files with proper column names"""
    
    # Create features directory
    features_dir = os.path.join(save_path, "extracted_features")
    os.makedirs(features_dir, exist_ok=True)
    
    # === TRAINING SET ===
    train_whole_df = train_whole.copy()
    train_whole_df['label'] = train_labels
    train_whole_df['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in train_labels]
    
    train_xai_df = train_xai.copy()
    train_xai_df['label'] = train_labels
    train_xai_df['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in train_labels]
    
    train_deep_df = train_deep.copy()
    train_deep_df['label'] = train_labels
    train_deep_df['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in train_labels]
    
    # Combined training features
    train_combined = pd.concat([train_whole, train_xai, train_deep], axis=1)
    train_combined['label'] = train_labels
    train_combined['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in train_labels]
    
    # === TEST SET ===
    test_whole_df = test_whole.copy()
    test_whole_df['label'] = test_labels
    test_whole_df['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in test_labels]
    
    test_xai_df = test_xai.copy()
    test_xai_df['label'] = test_labels
    test_xai_df['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in test_labels]
    
    test_deep_df = test_deep.copy()
    test_deep_df['label'] = test_labels
    test_deep_df['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in test_labels]
    
    # Combined test features
    test_combined = pd.concat([test_whole, test_xai, test_deep], axis=1)
    test_combined['label'] = test_labels
    test_combined['label_name'] = ['cancer' if l == 1 else 'non_cancer' for l in test_labels]
    
    # === SAVE TO EXCEL ===
    print("Saving features to Excel files...")
    
    train_whole_df.to_excel(os.path.join(features_dir, 'train_whole_image_radiomics.xlsx'), index=False)
    train_xai_df.to_excel(os.path.join(features_dir, 'train_xai_roi_radiomics.xlsx'), index=False)
    train_deep_df.to_excel(os.path.join(features_dir, 'train_deep_features.xlsx'), index=False)
    
    test_whole_df.to_excel(os.path.join(features_dir, 'test_whole_image_radiomics.xlsx'), index=False)
    test_xai_df.to_excel(os.path.join(features_dir, 'test_xai_roi_radiomics.xlsx'), index=False)
    test_deep_df.to_excel(os.path.join(features_dir, 'test_deep_features.xlsx'), index=False)
    
    train_combined.to_excel(os.path.join(features_dir, 'train_all_features_combined.xlsx'), index=False)
    test_combined.to_excel(os.path.join(features_dir, 'test_all_features_combined.xlsx'), index=False)
    
    # === SAVE TO CSV ===
    train_combined.to_csv(os.path.join(features_dir, 'train_all_features_combined.csv'), index=False)
    test_combined.to_csv(os.path.join(features_dir, 'test_all_features_combined.csv'), index=False)
    
    print("\n" + "="*60)
    print("FEATURES SAVED SUCCESSFULLY!")
    print("="*60)
    print(f"\nLocation: {features_dir}")
    print(f"\nFiles created:")
    print(f"  train_whole_image_radiomics.xlsx  ({train_whole_df.shape[1]-2} features)")
    print(f"  train_xai_roi_radiomics.xlsx      ({train_xai_df.shape[1]-2} features)")
    print(f"  train_deep_features.xlsx          ({train_deep_df.shape[1]-2} features)")
    print(f"  test_whole_image_radiomics.xlsx   ({test_whole_df.shape[1]-2} features)")
    print(f"  test_xai_roi_radiomics.xlsx       ({test_xai_df.shape[1]-2} features)")
    print(f"  test_deep_features.xlsx           ({test_deep_df.shape[1]-2} features)")
    print(f"  train_all_features_combined.xlsx  ({train_combined.shape[1]-2} total features)")
    print(f"  test_all_features_combined.xlsx   ({test_combined.shape[1]-2} total features)")
    print("="*60)
    
    return features_dir


# Run it
features_dir = save_features_to_excel(
    train_whole, train_xai, train_deep, train_labels_rad,
    test_whole, test_xai, test_deep, test_labels_rad,
    Config.OUTPUT_DIR
)

In [ ]:
# CELL 31: FEATURE FUSION AND MODEL TRAINING


In [ ]:
def train_fusion_models(train_features_dict, train_labels, test_features_dict, test_labels):
    """
    Train and evaluate different fusion strategies
    """
    results = {}
    
    # Define feature combinations
    feature_sets = {
        'Deep Only': train_features_dict['deep'],
        'Whole Radiomics': train_features_dict['whole'],
        'XAI-ROI Radiomics': train_features_dict['xai'],
        'Deep + Whole Radiomics': pd.concat([train_features_dict['deep'], train_features_dict['whole']], axis=1),
        'Deep + XAI Radiomics': pd.concat([train_features_dict['deep'], train_features_dict['xai']], axis=1),
        'All Features': pd.concat([train_features_dict['deep'], train_features_dict['whole'], 
                                   train_features_dict['xai']], axis=1)
    }
    
    test_feature_sets = {
        'Deep Only': test_features_dict['deep'],
        'Whole Radiomics': test_features_dict['whole'],
        'XAI-ROI Radiomics': test_features_dict['xai'],
        'Deep + Whole Radiomics': pd.concat([test_features_dict['deep'], test_features_dict['whole']], axis=1),
        'Deep + XAI Radiomics': pd.concat([test_features_dict['deep'], test_features_dict['xai']], axis=1),
        'All Features': pd.concat([test_features_dict['deep'], test_features_dict['whole'], 
                                   test_features_dict['xai']], axis=1)
    }
    
    print("\n" + "="*70)
    print("TRAINING FUSION MODELS")
    print("="*70)
    
    for name, X_train in feature_sets.items():
        print(f"\n{name}:")
        print(f"  Features: {X_train.shape[1]}")
        
        X_test = test_feature_sets[name]
        
        # Handle NaN values
        X_train = X_train.fillna(0)
        X_test = X_test.fillna(0)
        
        # Standardize features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Train XGBoost classifier
        clf = xgb.XGBClassifier(
            n_estimators=100,
            max_depth=5,
            learning_rate=0.1,
            random_state=42,
            use_label_encoder=False,
            eval_metric='logloss'
        )
        
        clf.fit(X_train_scaled, train_labels)
        
        # Evaluate
        y_pred = clf.predict(X_test_scaled)
        y_prob = clf.predict_proba(X_test_scaled)[:, 1]
        
        acc = accuracy_score(test_labels, y_pred)
        prec = precision_score(test_labels, y_pred, zero_division=0)
        rec = recall_score(test_labels, y_pred, zero_division=0)
        f1 = f1_score(test_labels, y_pred, zero_division=0)
        
        try:
            auc = roc_auc_score(test_labels, y_prob)
        except:
            auc = 0.5
        
        results[name] = {
            'accuracy': acc,
            'precision': prec,
            'recall': rec,
            'f1': f1,
            'auc': auc,
            'predictions': y_pred,
            'probabilities': y_prob,
            'model': clf,
            'scaler': scaler
        }
        
        print(f"  Accuracy: {acc*100:.2f}%")
        print(f"  F1-Score: {f1*100:.2f}%")
        print(f"  AUC: {auc:.4f}")
    
    return results


# Prepare feature dictionaries
train_features = {
    'deep': train_deep,
    'whole': train_whole,
    'xai': train_xai
}

test_features = {
    'deep': test_deep,
    'whole': test_whole,
    'xai': test_xai
}

# Train fusion models
fusion_results = train_fusion_models(train_features, train_labels_rad, test_features, test_labels_rad)


In [ ]:
# CELL 32: VISUALIZE FUSION RESULTS


In [ ]:
def plot_fusion_comparison(results, save_path=None):
    """Compare performance of different fusion strategies"""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    methods = list(results.keys())
    metrics = ['accuracy', 'precision', 'recall', 'f1', 'auc']
    
    # Bar plot
    x = np.arange(len(methods))
    width = 0.15
    
    colors = plt.cm.Set3(np.linspace(0, 1, len(metrics)))
    
    for i, metric in enumerate(metrics):
        values = [results[m][metric] for m in methods]
        axes[0].bar(x + i*width, values, width, label=metric.upper(), color=colors[i])
    
    axes[0].set_xlabel('Feature Set', fontsize=12)
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].set_title('Performance Comparison of Fusion Strategies', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x + width * 2)
    axes[0].set_xticklabels(methods, rotation=45, ha='right')
    axes[0].legend(loc='upper left')
    axes[0].set_ylim(0, 1.1)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Heatmap
    data = [[results[m][metric] for metric in metrics] for m in methods]
    im = axes[1].imshow(data, cmap='YlGnBu', aspect='auto', vmin=0, vmax=1)
    
    axes[1].set_xticks(range(len(metrics)))
    axes[1].set_xticklabels([m.upper() for m in metrics])
    axes[1].set_yticks(range(len(methods)))
    axes[1].set_yticklabels(methods)
    axes[1].set_title('Performance Heatmap', fontsize=14, fontweight='bold')
    
    # Add values to heatmap
    for i in range(len(methods)):
        for j in range(len(metrics)):
            text = axes[1].text(j, i, f'{data[i][j]:.2f}', ha='center', va='center', 
                               color='white' if data[i][j] > 0.5 else 'black', fontsize=10)
    
    plt.colorbar(im, ax=axes[1])
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()


plot_fusion_comparison(fusion_results, save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'fusion_comparison.png'))


In [ ]:
# CELL 33: XAI vs WHOLE IMAGE RADIOMICS


In [ ]:
def compare_xai_vs_whole_radiomics(train_xai, train_whole, test_xai, test_whole, 
                                   train_labels, test_labels, save_path=None):
    """
    Direct comparison of XAI-ROI vs Whole-Image radiomics
    """
    
    print("\n" + "="*60)
    print("XAI-ROI vs WHOLE-IMAGE RADIOMICS COMPARISON")
    print("="*60)
    
    # Train classifiers
    results = {}
    
    for name, X_train, X_test in [('Whole Image', train_whole, test_whole),
                                   ('XAI-ROI', train_xai, test_xai)]:
        X_train = X_train.fillna(0)
        X_test = X_test.fillna(0)
        
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        clf = xgb.XGBClassifier(n_estimators=100, max_depth=5, random_state=42,
                                use_label_encoder=False, eval_metric='logloss')
        clf.fit(X_train_scaled, train_labels)
        
        y_pred = clf.predict(X_test_scaled)
        y_prob = clf.predict_proba(X_test_scaled)[:, 1]
        
        results[name] = {
            'accuracy': accuracy_score(test_labels, y_pred),
            'f1': f1_score(test_labels, y_pred, zero_division=0),
            'auc': roc_auc_score(test_labels, y_prob) if len(np.unique(test_labels)) > 1 else 0.5,
            'predictions': y_pred,
            'probabilities': y_prob,
            'feature_importance': clf.feature_importances_,
            'feature_names': list(X_train.columns)
        }
    
    # Plot comparison
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Metrics comparison
    metrics = ['accuracy', 'f1', 'auc']
    x = np.arange(len(metrics))
    width = 0.35
    
    whole_values = [results['Whole Image'][m] for m in metrics]
    xai_values = [results['XAI-ROI'][m] for m in metrics]
    
    axes[0].bar(x - width/2, whole_values, width, label='Whole Image', color='steelblue')
    axes[0].bar(x + width/2, xai_values, width, label='XAI-ROI', color='coral')
    
    axes[0].set_ylabel('Score', fontsize=12)
    axes[0].set_title('Performance Comparison', fontsize=14, fontweight='bold')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels([m.upper() for m in metrics])
    axes[0].legend()
    axes[0].set_ylim(0, 1.1)
    axes[0].grid(True, alpha=0.3, axis='y')
    
    # Add improvement annotation
    for i, m in enumerate(metrics):
        improvement = (xai_values[i] - whole_values[i]) * 100
        color = 'green' if improvement > 0 else 'red'
        axes[0].annotate(f'{improvement:+.1f}%', 
                        xy=(x[i], max(whole_values[i], xai_values[i]) + 0.05),
                        ha='center', fontsize=10, color=color)
    
    # ROC curves
    for name, color in [('Whole Image', 'steelblue'), ('XAI-ROI', 'coral')]:
        fpr, tpr, _ = roc_curve(test_labels, results[name]['probabilities'])
        axes[1].plot(fpr, tpr, color=color, linewidth=2, 
                    label=f'{name} (AUC={results[name]["auc"]:.3f})')
    
    axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1)
    axes[1].set_xlabel('False Positive Rate', fontsize=12)
    axes[1].set_ylabel('True Positive Rate', fontsize=12)
    axes[1].set_title('ROC Curve Comparison', fontsize=14, fontweight='bold')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Feature importance (top 10)
    xai_importance = pd.Series(results['XAI-ROI']['feature_importance'],
                               index=results['XAI-ROI']['feature_names'])
    top_features = xai_importance.nlargest(10)
    
    axes[2].barh(range(len(top_features)), top_features.values, color='coral')
    axes[2].set_yticks(range(len(top_features)))
    axes[2].set_yticklabels([f.replace('xai_', '') for f in top_features.index])
    axes[2].set_xlabel('Importance', fontsize=12)
    axes[2].set_title('Top 10 XAI-ROI Features', fontsize=14, fontweight='bold')
    axes[2].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print summary
    print("\nSummary:")
    print(f"  Whole Image AUC: {results['Whole Image']['auc']:.4f}")
    print(f"  XAI-ROI AUC:     {results['XAI-ROI']['auc']:.4f}")
    improvement = (results['XAI-ROI']['auc'] - results['Whole Image']['auc']) * 100
    print(f"  Improvement:     {improvement:+.2f}%")
    
    return results


xai_vs_whole_results = compare_xai_vs_whole_radiomics(
    train_xai, train_whole, test_xai, test_whole,
    train_labels_rad, test_labels_rad,
    save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'xai_vs_whole_comparison.png')
)

print("\n" + "="*50)
print("PART 5 COMPLETE: Explanation-Guided Radiomics")
print("="*50)


In [ ]:
"""
============================================================================
PART 6: ROBUSTNESS TESTING AND FINAL SUMMARY
============================================================================
"""


In [ ]:
# CELL 34: ROBUSTNESS TESTING


In [ ]:
class RobustnessEvaluator:
    """Evaluate model robustness under perturbations"""
    
    def __init__(self, model, device):
        self.model = model
        self.device = device
    
    def add_gaussian_noise(self, image, sigma=0.1):
        """Add Gaussian noise"""
        noise = torch.randn_like(image) * sigma
        return torch.clamp(image + noise, 0, 1)
    
    def add_blur(self, image, kernel_size=5):
        """Add Gaussian blur"""
        img_np = image.squeeze().permute(1, 2, 0).cpu().numpy()
        blurred = cv2.GaussianBlur(img_np, (kernel_size, kernel_size), 0)
        return torch.tensor(blurred).permute(2, 0, 1).unsqueeze(0).to(self.device)
    
    def add_occlusion(self, image, size=32, num_patches=3):
        """Add random occlusions"""
        img = image.clone()
        _, _, h, w = img.shape
        
        for _ in range(num_patches):
            x = random.randint(0, w - size)
            y = random.randint(0, h - size)
            img[:, :, y:y+size, x:x+size] = 0
        
        return img
    
    def evaluate_robustness(self, test_loader, perturbation_types=['noise', 'blur', 'occlusion']):
        """Evaluate model robustness"""
        
        self.model.eval()
        results = {'clean': {'correct': 0, 'total': 0}}
        
        for p_type in perturbation_types:
            results[p_type] = {'correct': 0, 'total': 0}
        
        for images, labels, _ in tqdm(test_loader, desc="Testing robustness"):
            images = images.to(self.device)
            labels = labels.to(self.device)
            
            # Clean accuracy
            with torch.no_grad():
                outputs = self.model(images)
                _, preds = outputs.max(1)
                results['clean']['correct'] += preds.eq(labels).sum().item()
                results['clean']['total'] += labels.size(0)
            
            # Perturbation accuracy
            for p_type in perturbation_types:
                for i in range(images.size(0)):
                    img = images[i:i+1]
                    
                    if p_type == 'noise':
                        perturbed = self.add_gaussian_noise(img, sigma=0.15)
                    elif p_type == 'blur':
                        perturbed = self.add_blur(img, kernel_size=7)
                    else:  # occlusion
                        perturbed = self.add_occlusion(img, size=32, num_patches=2)
                    
                    with torch.no_grad():
                        output = self.model(perturbed)
                        _, pred = output.max(1)
                        results[p_type]['correct'] += pred.eq(labels[i]).item()
                        results[p_type]['total'] += 1
        
        # Calculate accuracies
        for key in results:
            results[key]['accuracy'] = results[key]['correct'] / results[key]['total'] if results[key]['total'] > 0 else 0
        
        return results


def plot_robustness_results(results, save_path=None):
    """Plot robustness evaluation results"""
    
    conditions = list(results.keys())
    accuracies = [results[c]['accuracy'] * 100 for c in conditions]
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    colors = ['#2ecc71', '#e74c3c', '#3498db', '#9b59b6']
    bars = ax.bar(conditions, accuracies, color=colors[:len(conditions)], edgecolor='black', linewidth=1.5)
    
    # Add value labels
    for bar, acc in zip(bars, accuracies):
        height = bar.get_height()
        ax.annotate(f'{acc:.1f}%',
                   xy=(bar.get_x() + bar.get_width() / 2, height),
                   xytext=(0, 3),
                   textcoords="offset points",
                   ha='center', va='bottom', fontsize=12, fontweight='bold')
    
    ax.set_ylabel('Accuracy (%)', fontsize=12)
    ax.set_title('Model Robustness Under Perturbations', fontsize=14, fontweight='bold')
    ax.set_ylim(0, 110)
    ax.grid(True, alpha=0.3, axis='y')
    
    # Add drop indicators
    clean_acc = results['clean']['accuracy'] * 100
    for i, (cond, acc) in enumerate(zip(conditions[1:], accuracies[1:]), 1):
        drop = clean_acc - acc
        ax.annotate(f'↓{drop:.1f}%', xy=(i, acc - 5), ha='center', fontsize=10, color='red')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    
    # Print summary
    print("\n" + "="*50)
    print("ROBUSTNESS TEST SUMMARY")
    print("="*50)
    for cond in conditions:
        acc = results[cond]['accuracy'] * 100
        if cond == 'clean':
            print(f"{cond.capitalize():<15} {acc:.2f}%")
        else:
            drop = clean_acc - acc
            print(f"{cond.capitalize():<15} {acc:.2f}% (↓{drop:.2f}%)")


# Run robustness evaluation
print("\nEvaluating Model Robustness...")
robustness_evaluator = RobustnessEvaluator(model, device)
robustness_results = robustness_evaluator.evaluate_robustness(test_loader)

plot_robustness_results(robustness_results, 
                       save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'robustness_results.png'))


In [ ]:
# CELL 35: GENERATE COMPREHENSIVE SUMMARY


In [ ]:
def generate_final_summary(test_results, faithfulness_results, fusion_results, robustness_results):
    """Generate final study summary"""
    
    print("\n")
    print("="*80)
    print("  COMPREHENSIVE STUDY SUMMARY")
    print("  Explanation-Guided Radiomics with Multi-Method XAI Validation")
    print("  for Colon Cancer Classification in Abdominal CT")
    print("="*80)
    
    print("\n" + "-"*80)
    print("1. BASELINE MODEL PERFORMANCE")
    print("-"*80)
    print(f"   Model: {Config.MODEL_NAME}")
    print(f"   Accuracy:    {test_results['accuracy']*100:.2f}%")
    print(f"   Sensitivity: {test_results['recall']*100:.2f}%")
    print(f"   Specificity: {test_results['specificity']*100:.2f}%")
    print(f"   F1-Score:    {test_results['f1']*100:.2f}%")
    print(f"   AUC-ROC:     {test_results['auc']:.4f}")
    
    print("\n" + "-"*80)
    print("2. XAI FAITHFULNESS EVALUATION")
    print("-"*80)
    methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
    labels = ['Grad-CAM++', 'Integrated Gradients', 'SHAP', 'LIME']
    
    print(f"   {'Method':<25} {'Deletion AUC ↓':<18} {'Insertion AUC ↑':<18}")
    print("   " + "-"*60)
    
    best_del = min(methods, key=lambda m: np.mean(faithfulness_results[m]['deletion_aucs']))
    best_ins = max(methods, key=lambda m: np.mean(faithfulness_results[m]['insertion_aucs']))
    
    for method, label in zip(methods, labels):
        del_auc = np.mean(faithfulness_results[method]['deletion_aucs'])
        ins_auc = np.mean(faithfulness_results[method]['insertion_aucs'])
        
        del_mark = " ★" if method == best_del else ""
        ins_mark = " ★" if method == best_ins else ""
        
        print(f"   {label:<25} {del_auc:.4f}{del_mark:<10} {ins_auc:.4f}{ins_mark:<10}")
    
    print(f"\n   Best Deletion: {labels[methods.index(best_del)]} (lower = better)")
    print(f"   Best Insertion: {labels[methods.index(best_ins)]} (higher = better)")
    
    print("\n" + "-"*80)
    print("3. FEATURE FUSION COMPARISON")
    print("-"*80)
    
    print(f"   {'Feature Set':<25} {'Accuracy':<12} {'F1-Score':<12} {'AUC':<12}")
    print("   " + "-"*60)
    
    best_fusion = max(fusion_results.keys(), key=lambda k: fusion_results[k]['auc'])
    
    for name, res in fusion_results.items():
        mark = " ★" if name == best_fusion else ""
        print(f"   {name:<25} {res['accuracy']*100:>6.2f}%     {res['f1']*100:>6.2f}%     {res['auc']:.4f}{mark}")
    
    print(f"\n   Best Fusion Strategy: {best_fusion}")
    
    print("\n" + "-"*80)
    print("4. ROBUSTNESS EVALUATION")
    print("-"*80)
    
    clean_acc = robustness_results['clean']['accuracy'] * 100
    print(f"   Clean Accuracy: {clean_acc:.2f}%")
    
    avg_drop = 0
    count = 0
    for cond in robustness_results:
        if cond != 'clean':
            acc = robustness_results[cond]['accuracy'] * 100
            drop = clean_acc - acc
            avg_drop += drop
            count += 1
            print(f"   {cond.capitalize()}: {acc:.2f}% (drop: {drop:.2f}%)")
    
    avg_drop /= count if count > 0 else 1
    print(f"\n   Average Accuracy Drop: {avg_drop:.2f}%")
    
    print("\n" + "-"*80)
    print("5. KEY FINDINGS")
    print("-"*80)
    
    # Determine XAI vs Whole improvement
    if 'XAI-ROI Radiomics' in fusion_results and 'Whole Radiomics' in fusion_results:
        xai_auc = fusion_results['XAI-ROI Radiomics']['auc']
        whole_auc = fusion_results['Whole Radiomics']['auc']
        improvement = (xai_auc - whole_auc) * 100
        
        if improvement > 0:
            print(f"   ✓ XAI-guided radiomics improves AUC by {improvement:.2f}% vs whole-image")
        else:
            print(f"   • Whole-image radiomics performed {-improvement:.2f}% better than XAI-ROI")
    
    if best_fusion.startswith('Deep + XAI'):
        print(f"   ✓ Deep + XAI fusion achieves best overall performance")
    
    if avg_drop < 10:
        print(f"   ✓ Model shows good robustness (avg accuracy drop < 10%)")
    
    print(f"   ✓ {labels[methods.index(best_del)]} provides most faithful explanations")
    
    print("\n" + "="*80)
    print("  END OF SUMMARY")
    print("="*80)


# Generate summary
generate_final_summary(test_results, faithfulness_results, fusion_results, robustness_results)


In [ ]:
# CELL 36: SAVE ALL RESULTS


In [ ]:
def save_all_results(config, test_results, faithfulness_results, fusion_results, robustness_results):
    """Save all results to files"""
    
    results_summary = {
        'model_name': config.MODEL_NAME,
        'date': datetime.now().isoformat(),
        'test_metrics': {
            'accuracy': test_results['accuracy'],
            'precision': test_results['precision'],
            'recall': test_results['recall'],
            'specificity': test_results['specificity'],
            'f1': test_results['f1'],
            'auc': test_results['auc']
        },
        'faithfulness': {
            method: {
                'deletion_auc_mean': float(np.mean(faithfulness_results[method]['deletion_aucs'])),
                'deletion_auc_std': float(np.std(faithfulness_results[method]['deletion_aucs'])),
                'insertion_auc_mean': float(np.mean(faithfulness_results[method]['insertion_aucs'])),
                'insertion_auc_std': float(np.std(faithfulness_results[method]['insertion_aucs']))
            }
            for method in ['gradcam++', 'integrated_gradients', 'shap', 'lime']
        },
        'fusion_results': {
            name: {
                'accuracy': res['accuracy'],
                'f1': res['f1'],
                'auc': res['auc']
            }
            for name, res in fusion_results.items()
        },
        'robustness': {
            cond: {'accuracy': res['accuracy']}
            for cond, res in robustness_results.items()
        }
    }
    
    # Save as JSON
    results_path = os.path.join(config.OUTPUT_DIR, 'results_summary.json')
    with open(results_path, 'w') as f:
        json.dump(results_summary, f, indent=2)
    
    print(f"\nResults saved to: {results_path}")
    
    # Save feature importance
    if 'Deep + XAI Radiomics' in fusion_results:
        model_obj = fusion_results['Deep + XAI Radiomics'].get('model')
        if model_obj is not None:
            importance_df = pd.DataFrame({
                'feature': range(len(model_obj.feature_importances_)),
                'importance': model_obj.feature_importances_
            }).sort_values('importance', ascending=False)
            importance_df.to_csv(os.path.join(config.OUTPUT_DIR, 'feature_importance.csv'), index=False)
    
    return results_summary


# Save results
results_summary = save_all_results(Config, test_results, faithfulness_results, fusion_results, robustness_results)


In [ ]:
# CELL 37: CREATE PUBLICATION-READY FIGURES


In [ ]:
def create_publication_figure(test_results, faithfulness_results, fusion_results, save_path=None):
    """Create a comprehensive publication-ready figure"""
    
    fig = plt.figure(figsize=(20, 16))
    
    # Create grid
    gs = fig.add_gridspec(3, 3, hspace=0.3, wspace=0.3)
    
    # 1. Confusion Matrix
    ax1 = fig.add_subplot(gs[0, 0])
    cm = confusion_matrix(test_results['labels'], test_results['predictions'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1)
    ax1.set_xlabel('Predicted')
    ax1.set_ylabel('Actual')
    ax1.set_title('A) Confusion Matrix', fontweight='bold')
    
    # 2. ROC Curve
    ax2 = fig.add_subplot(gs[0, 1])
    fpr, tpr, _ = roc_curve(test_results['labels'], test_results['probabilities'])
    ax2.plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {test_results["auc"]:.3f}')
    ax2.plot([0, 1], [0, 1], 'k--')
    ax2.set_xlabel('False Positive Rate')
    ax2.set_ylabel('True Positive Rate')
    ax2.set_title('B) ROC Curve', fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    # 3. Faithfulness Comparison
    ax3 = fig.add_subplot(gs[0, 2])
    methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
    labels_short = ['GradCAM++', 'IG', 'SHAP', 'LIME']
    
    del_aucs = [np.mean(faithfulness_results[m]['deletion_aucs']) for m in methods]
    ins_aucs = [np.mean(faithfulness_results[m]['insertion_aucs']) for m in methods]
    
    x = np.arange(len(methods))
    width = 0.35
    
    ax3.bar(x - width/2, del_aucs, width, label='Deletion ↓', color='coral')
    ax3.bar(x + width/2, ins_aucs, width, label='Insertion ↑', color='steelblue')
    ax3.set_ylabel('AUC')
    ax3.set_xticks(x)
    ax3.set_xticklabels(labels_short)
    ax3.set_title('C) XAI Faithfulness', fontweight='bold')
    ax3.legend()
    ax3.grid(True, alpha=0.3, axis='y')
    
    # 4. Deletion Curves
    ax4 = fig.add_subplot(gs[1, 0])
    colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']
    steps = np.linspace(0, 100, len(faithfulness_results[methods[0]]['deletion_scores'][0]))
    
    for method, color, label in zip(methods, colors, labels_short):
        scores = np.array(faithfulness_results[method]['deletion_scores'])
        mean_scores = scores.mean(axis=0)
        ax4.plot(steps, mean_scores, color=color, linewidth=2, label=label)
    
    ax4.set_xlabel('Pixels Removed (%)')
    ax4.set_ylabel('Probability')
    ax4.set_title('D) Deletion Curves', fontweight='bold')
    ax4.legend()
    ax4.grid(True, alpha=0.3)
    
    # 5. Insertion Curves
    ax5 = fig.add_subplot(gs[1, 1])
    
    for method, color, label in zip(methods, colors, labels_short):
        scores = np.array(faithfulness_results[method]['insertion_scores'])
        mean_scores = scores.mean(axis=0)
        ax5.plot(steps, mean_scores, color=color, linewidth=2, label=label)
    
    ax5.set_xlabel('Pixels Inserted (%)')
    ax5.set_ylabel('Probability')
    ax5.set_title('E) Insertion Curves', fontweight='bold')
    ax5.legend()
    ax5.grid(True, alpha=0.3)
    
    # 6. Fusion Performance
    ax6 = fig.add_subplot(gs[1, 2])
    fusion_names = list(fusion_results.keys())
    fusion_aucs = [fusion_results[n]['auc'] for n in fusion_names]
    
    colors_fusion = plt.cm.viridis(np.linspace(0.2, 0.8, len(fusion_names)))
    bars = ax6.barh(range(len(fusion_names)), fusion_aucs, color=colors_fusion)
    ax6.set_yticks(range(len(fusion_names)))
    ax6.set_yticklabels(fusion_names, fontsize=9)
    ax6.set_xlabel('AUC')
    ax6.set_title('F) Fusion Strategies', fontweight='bold')
    ax6.set_xlim(0, 1)
    ax6.grid(True, alpha=0.3, axis='x')
    
    for bar, auc in zip(bars, fusion_aucs):
        ax6.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2, 
                f'{auc:.3f}', va='center', fontsize=9)
    
    # 7-9. Performance metrics table
    ax7 = fig.add_subplot(gs[2, :])
    ax7.axis('off')
    
    # Create summary table
    table_data = [
        ['Metric', 'Value', 'Description'],
        ['Accuracy', f"{test_results['accuracy']*100:.2f}%", 'Overall classification accuracy'],
        ['Sensitivity', f"{test_results['recall']*100:.2f}%", 'True positive rate (cancer detection)'],
        ['Specificity', f"{test_results['specificity']*100:.2f}%", 'True negative rate'],
        ['F1-Score', f"{test_results['f1']*100:.2f}%", 'Harmonic mean of precision and recall'],
        ['AUC-ROC', f"{test_results['auc']:.4f}", 'Area under ROC curve'],
        ['Best XAI', 'Grad-CAM++', 'Most faithful explanation method'],
        ['Best Fusion', max(fusion_results.keys(), key=lambda k: fusion_results[k]['auc']), 
         f"AUC: {max(fusion_results.values(), key=lambda x: x['auc'])['auc']:.4f}"]
    ]
    
    table = ax7.table(cellText=table_data, cellLoc='center', loc='center',
                      colWidths=[0.2, 0.15, 0.4])
    table.auto_set_font_size(False)
    table.set_fontsize(11)
    table.scale(1.2, 1.8)
    
    # Style header
    for i in range(3):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(color='white', fontweight='bold')
    
    ax7.set_title('G) Summary of Results', fontweight='bold', fontsize=14, pad=20)
    
    plt.suptitle('Explanation-Guided Radiomics with Multi-Method XAI Validation\nfor Colon Cancer Classification',
                fontsize=16, fontweight='bold', y=0.98)
    
    if save_path:
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()


# Create publication figure
create_publication_figure(
    test_results, faithfulness_results, fusion_results,
    save_path=os.path.join(Config.FIGURE_SAVE_PATH, 'publication_figure.png')
)


print("\n" + "="*80)
print("  STUDY COMPLETE!")
print("="*80)
print(f"\nAll results and figures saved to: {Config.OUTPUT_DIR}")
print("\nGenerated files:")
print(f"  - {Config.FIGURE_SAVE_PATH}/sample_images.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/augmentation_examples.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/training_history.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/evaluation_metrics.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/xai_comparison.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/faithfulness_results.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/sanity_check.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/fusion_comparison.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/xai_vs_whole_comparison.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/robustness_results.png")
print(f"  - {Config.FIGURE_SAVE_PATH}/publication_figure.png")
print(f"  - {Config.OUTPUT_DIR}/results_summary.json")
print(f"  - {Config.MODEL_SAVE_PATH}/best_model_{Config.MODEL_NAME}.pth")
print("="*80)


In [ ]:
# Figure 1: Confusion Matrix
fig, ax = plt.subplots(figsize=(8, 6))
cm = confusion_matrix(test_results['labels'], test_results['predictions'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
            xticklabels=['Non-Cancer', 'Cancer'],
            yticklabels=['Non-Cancer', 'Cancer'],
            annot_kws={'size': 16})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Confusion Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig1_confusion_matrix.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 2: ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
fpr, tpr, _ = roc_curve(test_results['labels'], test_results['probabilities'])
ax.plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {test_results["auc"]:.4f}')
ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random Classifier')
ax.fill_between(fpr, tpr, alpha=0.3)
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('ROC Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig2_roc_curve.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 3: Precision-Recall Curve
fig, ax = plt.subplots(figsize=(8, 6))
precision_vals, recall_vals, _ = precision_recall_curve(test_results['labels'], test_results['probabilities'])
ap = average_precision_score(test_results['labels'], test_results['probabilities'])
ax.plot(recall_vals, precision_vals, 'g-', linewidth=2, label=f'AP = {ap:.4f}')
ax.fill_between(recall_vals, precision_vals, alpha=0.3, color='green')
ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title('Precision-Recall Curve', fontsize=14, fontweight='bold')
ax.legend(loc='lower left', fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig3_precision_recall_curve.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 4: XAI Faithfulness Comparison
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
labels_short = ['Grad-CAM++', 'Integrated\nGradients', 'SHAP', 'LIME']

del_aucs = [np.mean(faithfulness_results[m]['deletion_aucs']) for m in methods]
ins_aucs = [np.mean(faithfulness_results[m]['insertion_aucs']) for m in methods]

x = np.arange(len(methods))
width = 0.35

bars1 = ax.bar(x - width/2, del_aucs, width, label='Deletion AUC (↓ lower is better)', color='coral', edgecolor='black')
bars2 = ax.bar(x + width/2, ins_aucs, width, label='Insertion AUC (↑ higher is better)', color='steelblue', edgecolor='black')

ax.set_ylabel('AUC', fontsize=12)
ax.set_xlabel('XAI Method', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(labels_short, fontsize=11)
ax.set_title('XAI Faithfulness Evaluation', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

# Add value labels on bars
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, 
            f'{bar.get_height():.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig4_xai_faithfulness_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 5: Deletion Curves
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
labels_short = ['Grad-CAM++', 'Integrated Gradients', 'SHAP', 'LIME']
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

steps = np.linspace(0, 100, len(faithfulness_results[methods[0]]['deletion_scores'][0]))

for method, color, label in zip(methods, colors, labels_short):
    scores = np.array(faithfulness_results[method]['deletion_scores'])
    mean_scores = scores.mean(axis=0)
    std_scores = scores.std(axis=0)
    ax.plot(steps, mean_scores, color=color, linewidth=2, label=label)
    ax.fill_between(steps, mean_scores - std_scores, mean_scores + std_scores, color=color, alpha=0.2)

ax.set_xlabel('Pixels Removed (%)', fontsize=12)
ax.set_ylabel('Cancer Probability', fontsize=12)
ax.set_title('Deletion Test (Lower AUC = More Faithful)', fontsize=14, fontweight='bold')
ax.legend(loc='upper right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 100])
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig5_deletion_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 6: Insertion Curves
fig, ax = plt.subplots(figsize=(10, 6))
methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
labels_short = ['Grad-CAM++', 'Integrated Gradients', 'SHAP', 'LIME']
colors = ['#e41a1c', '#377eb8', '#4daf4a', '#984ea3']

steps = np.linspace(0, 100, len(faithfulness_results[methods[0]]['insertion_scores'][0]))

for method, color, label in zip(methods, colors, labels_short):
    scores = np.array(faithfulness_results[method]['insertion_scores'])
    mean_scores = scores.mean(axis=0)
    std_scores = scores.std(axis=0)
    ax.plot(steps, mean_scores, color=color, linewidth=2, label=label)
    ax.fill_between(steps, mean_scores - std_scores, mean_scores + std_scores, color=color, alpha=0.2)

ax.set_xlabel('Pixels Inserted (%)', fontsize=12)
ax.set_ylabel('Cancer Probability', fontsize=12)
ax.set_title('Insertion Test (Higher AUC = More Faithful)', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 100])
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig6_insertion_curves.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 7: Fusion Strategies Comparison
fig, ax = plt.subplots(figsize=(12, 7))
fusion_names = list(fusion_results.keys())
fusion_aucs = [fusion_results[n]['auc'] for n in fusion_names]
fusion_f1s = [fusion_results[n]['f1'] for n in fusion_names]
fusion_accs = [fusion_results[n]['accuracy'] for n in fusion_names]

x = np.arange(len(fusion_names))
width = 0.25

bars1 = ax.bar(x - width, fusion_accs, width, label='Accuracy', color='#2ecc71', edgecolor='black')
bars2 = ax.bar(x, fusion_f1s, width, label='F1-Score', color='#3498db', edgecolor='black')
bars3 = ax.bar(x + width, fusion_aucs, width, label='AUC', color='#e74c3c', edgecolor='black')

ax.set_ylabel('Score', fontsize=12)
ax.set_xlabel('Feature Fusion Strategy', fontsize=12)
ax.set_xticks(x)
ax.set_xticklabels(fusion_names, rotation=45, ha='right', fontsize=10)
ax.set_title('Comparison of Feature Fusion Strategies', fontsize=14, fontweight='bold')
ax.legend(loc='lower right', fontsize=10)
ax.grid(True, alpha=0.3, axis='y')
ax.set_ylim([0, 1.1])

plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig7_fusion_comparison.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 8: Fusion AUC Horizontal Bar Chart
fig, ax = plt.subplots(figsize=(10, 6))
fusion_names = list(fusion_results.keys())
fusion_aucs = [fusion_results[n]['auc'] for n in fusion_names]

# Sort by AUC
sorted_indices = np.argsort(fusion_aucs)
fusion_names_sorted = [fusion_names[i] for i in sorted_indices]
fusion_aucs_sorted = [fusion_aucs[i] for i in sorted_indices]

colors_fusion = plt.cm.RdYlGn(np.linspace(0.2, 0.8, len(fusion_names)))
bars = ax.barh(range(len(fusion_names_sorted)), fusion_aucs_sorted, color=colors_fusion, edgecolor='black')

ax.set_yticks(range(len(fusion_names_sorted)))
ax.set_yticklabels(fusion_names_sorted, fontsize=11)
ax.set_xlabel('AUC', fontsize=12)
ax.set_title('Feature Fusion Strategies Ranked by AUC', fontsize=14, fontweight='bold')
ax.set_xlim([0, 1.1])
ax.grid(True, alpha=0.3, axis='x')

for bar, auc in zip(bars, fusion_aucs_sorted):
    ax.text(bar.get_width() + 0.02, bar.get_y() + bar.get_height()/2, 
            f'{auc:.4f}', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig8_fusion_auc_ranking.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Figure 9: Summary Metrics Table
fig, ax = plt.subplots(figsize=(12, 6))
ax.axis('off')

# Find best XAI method
methods = ['gradcam++', 'integrated_gradients', 'shap', 'lime']
best_del = min(methods, key=lambda m: np.mean(faithfulness_results[m]['deletion_aucs']))
best_ins = max(methods, key=lambda m: np.mean(faithfulness_results[m]['insertion_aucs']))

table_data = [
    ['Metric', 'Value', 'Description'],
    ['Accuracy', f"{test_results['accuracy']*100:.2f}%", 'Overall classification accuracy'],
    ['Sensitivity', f"{test_results['recall']*100:.2f}%", 'True positive rate (cancer detection)'],
    ['Specificity', f"{test_results['specificity']*100:.2f}%", 'True negative rate'],
    ['Precision', f"{test_results['precision']*100:.2f}%", 'Positive predictive value'],
    ['F1-Score', f"{test_results['f1']*100:.2f}%", 'Harmonic mean of precision and recall'],
    ['AUC-ROC', f"{test_results['auc']:.4f}", 'Area under ROC curve'],
    ['Best XAI (Deletion)', best_del, f"AUC: {np.mean(faithfulness_results[best_del]['deletion_aucs']):.4f}"],
    ['Best XAI (Insertion)', best_ins, f"AUC: {np.mean(faithfulness_results[best_ins]['insertion_aucs']):.4f}"],
    ['Best Fusion', max(fusion_results.keys(), key=lambda k: fusion_results[k]['auc']), 
     f"AUC: {max(fusion_results.values(), key=lambda x: x['auc'])['auc']:.4f}"]
]

table = ax.table(cellText=table_data, cellLoc='center', loc='center',
                  colWidths=[0.25, 0.2, 0.45])
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 2.0)

# Style header
for i in range(3):
    table[(0, i)].set_facecolor('#2c3e50')
    table[(0, i)].set_text_props(color='white', fontweight='bold')

# Alternate row colors
for i in range(1, len(table_data)):
    for j in range(3):
        if i % 2 == 0:
            table[(i, j)].set_facecolor('#ecf0f1')

ax.set_title('Summary of Results', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(Config.FIGURE_SAVE_PATH, 'fig9_summary_table.png'), dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Print summary of all saved figures
print("\n" + "="*60)
print("ALL FIGURES SAVED SUCCESSFULLY!")
print("="*60)
print(f"\nLocation: {Config.FIGURE_SAVE_PATH}")
print("\nFigures created:")
print("  1. fig1_confusion_matrix.png")
print("  2. fig2_roc_curve.png")
print("  3. fig3_precision_recall_curve.png")
print("  4. fig4_xai_faithfulness_comparison.png")
print("  5. fig5_deletion_curves.png")
print("  6. fig6_insertion_curves.png")
print("  7. fig7_fusion_comparison.png")
print("  8. fig8_fusion_auc_ranking.png")
print("  9. fig9_summary_table.png")
print("="*60)